In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0: CONFIGURATION & SETUP
# ═══════════════════════════════════════════════════════════════════════════════
# Env: omicverse
# ═══════════════════════════════════════════════════════════════════════════════
import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR    = Path('/fs/scratch/PAS2598/senescence_analysis')
DATASET     = 'morabito_dsad'

# Input
SPATIAL_H5AD = Path('/fs/ess/PDE0075/senescence/human/spatial/Morabito_DSAD/morabito_spatial.h5ad')
MARKERS_DIR  = Path('/fs/ess/PDE0075/senescence/human/markers')

# Output
DATA_DIR    = BASE_DIR / 'data' / '10_spatial'
FIGURES_DIR = BASE_DIR / 'figures' / '10_spatial' / DATASET
RESULTS_DIR = BASE_DIR / 'results' / '10_spatial' / DATASET

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── QC thresholds ────────────────────────────────────────────────────────────
MIN_COUNTS = 500
MIN_GENES  = 200
MAX_MT_PCT = 20

# ── Senescence parameters ────────────────────────────────────────────────────
SD_THRESHOLD  = 2.0
SENEPY_TISSUE = 'hippocampus'

# ── Load dataset ─────────────────────────────────────────────────────────────
print("=" * 60)
print(f"SPATIAL TRANSCRIPTOMICS PIPELINE")
print(f"Dataset: {DATASET}")
print("=" * 60)

adata_sp = sc.read_h5ad(SPATIAL_H5AD)
print(f"\n  Loaded: {adata_sp.shape[0]:,} spots × {adata_sp.shape[1]:,} genes")
print(f"  X dtype: {adata_sp.X.dtype}, sparse: {type(adata_sp.X).__name__}")

# ── Standardize column names ─────────────────────────────────────────────────
# Diagnosis groups
print(f"\n  Diagnosis groups:")
print(f"  {adata_sp.obs['Diagnosis'].value_counts().to_dict()}")

# Rename deconvolution columns to c2l_ prefix for consistency
deconv_map = {
    'ASC_deconv':  'c2l_Astrocyte',
    'EX_deconv':   'c2l_Excitatory',
    'INH_deconv':  'c2l_Inhibitory',
    'MG_deconv':   'c2l_Microglia',
    'ODC_deconv':  'c2l_Oligodendrocyte',
    'OPC_deconv':  'c2l_OPC',
    'VASC_deconv': 'c2l_Vascular',
}
for old, new in deconv_map.items():
    if old in adata_sp.obs.columns:
        adata_sp.obs[new] = adata_sp.obs[old]
print(f"\n  Deconvolution columns renamed: {list(deconv_map.values())}")

# Dominant cell type from deconvolution
c2l_cols = [c for c in adata_sp.obs.columns if c.startswith('c2l_')]
adata_sp.obs['dominant_celltype'] = adata_sp.obs[c2l_cols].idxmax(axis=1).str.replace('c2l_', '')
print(f"  Dominant cell types: {adata_sp.obs['dominant_celltype'].value_counts().to_dict()}")

# Build spatial coordinates into obsm['spatial'] from imagerow/imagecol
adata_sp.obsm['spatial'] = np.column_stack([
    adata_sp.obs['imagecol'].values.astype(float),
    adata_sp.obs['imagerow'].values.astype(float)
])
n_nan = np.isnan(adata_sp.obsm['spatial']).any(axis=1).sum()
print(f"\n  Spatial coords: obsm['spatial'] created from imagecol/imagerow")
print(f"  NaN spatial coords: {n_nan}")

# Standardize sample/group identifiers
adata_sp.obs['sample_id'] = adata_sp.obs['Sample'].astype(str)
adata_sp.obs['group'] = adata_sp.obs['Diagnosis'].astype(str)
adata_sp.obs['sex'] = adata_sp.obs['Sex'].astype(str)
adata_sp.obs['age'] = adata_sp.obs['Age'].astype(int)

# ── Sample summary ───────────────────────────────────────────────────────────
print(f"\n  Samples: {adata_sp.obs['sample_id'].nunique()}")
sample_summary = (adata_sp.obs.groupby(['sample_id', 'group', 'sex', 'age'])
                  .size().reset_index(name='n_spots')
                  .sort_values('group'))
for _, row in sample_summary.iterrows():
    print(f"    {row['sample_id']:<35s} {row['group']:<10s} {row['sex']:<3s} "
          f"age={row['age']:<3d} n={row['n_spots']}")

# ── Helper function ──────────────────────────────────────────────────────────
def get_meta(sample_id):
    mask = adata_sp.obs['sample_id'] == sample_id
    row = adata_sp.obs.loc[mask].iloc[0]
    return sample_id, row['group']

samples = sorted(adata_sp.obs['sample_id'].unique())
n_samples = len(samples)

# ── Print config ─────────────────────────────────────────────────────────────
print(f"\n  Output paths:")
print(f"    Data:    {DATA_DIR}")
print(f"    Figures: {FIGURES_DIR}")
print(f"    Results: {RESULTS_DIR}")
print(f"\n  QC: min_counts={MIN_COUNTS}, min_genes={MIN_GENES}, max_MT%={MAX_MT_PCT}")
print(f"  Senescence: {SENEPY_TISSUE}, threshold=mean+{SD_THRESHOLD}SD per sample")
print(f"  NOTE: Deconvolution already performed — skipping cell2location")
print(f"  Cell types: {sorted(deconv_map.values())}")
print("\n✓ Configuration ready")
print(f"  → Next: Cell 1 (QC) or Cell 8 (Preprocess) if skipping QC")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1: QUALITY CONTROL
# ═══════════════════════════════════════════════════════════════════════════════
# Env: omicverse
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("QUALITY CONTROL")
print("=" * 60)

print(f"\n  Before QC: {adata_sp.n_obs:,} spots × {adata_sp.n_vars:,} genes")

# ── Calculate QC metrics ─────────────────────────────────────────────────────
adata_sp.var['mt'] = adata_sp.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata_sp, qc_vars=['mt'], percent_top=None,
                           log1p=False, inplace=True)

print(f"\n  total_counts: mean={adata_sp.obs['total_counts'].mean():.0f}, "
      f"median={adata_sp.obs['total_counts'].median():.0f}")
print(f"  n_genes_by_counts: mean={adata_sp.obs['n_genes_by_counts'].mean():.0f}, "
      f"median={adata_sp.obs['n_genes_by_counts'].median():.0f}")
print(f"  pct_counts_mt: mean={adata_sp.obs['pct_counts_mt'].mean():.1f}%, "
      f"max={adata_sp.obs['pct_counts_mt'].max():.1f}%")

# ── Pre-QC distributions per group ──────────────────────────────────────────
print(f"\n── Pre-QC summary by diagnosis group ──\n")
print(f"  {'Group':<10s} {'n_spots':>8s} {'med_UMI':>10s} {'med_genes':>10s} {'med_MT%':>10s}")
print(f"  {'─'*52}")
for grp in ['Control', 'earlyAD', 'AD', 'AD_DS']:
    mask = adata_sp.obs['group'] == grp
    print(f"  {grp:<10s} {mask.sum():>8d} "
          f"{adata_sp.obs.loc[mask, 'total_counts'].median():>10.0f} "
          f"{adata_sp.obs.loc[mask, 'n_genes_by_counts'].median():>10.0f} "
          f"{adata_sp.obs.loc[mask, 'pct_counts_mt'].median():>10.1f}")

# ── Apply QC filters ─────────────────────────────────────────────────────────
print(f"\n── Applying QC filters ──")
print(f"  min_counts = {MIN_COUNTS}")
print(f"  min_genes  = {MIN_GENES}")
print(f"  max_MT%    = {MAX_MT_PCT}")

n_before = adata_sp.n_obs

filt_counts = adata_sp.obs['total_counts'] >= MIN_COUNTS
filt_genes  = adata_sp.obs['n_genes_by_counts'] >= MIN_GENES
filt_mt     = adata_sp.obs['pct_counts_mt'] <= MAX_MT_PCT

print(f"\n  Fail min_counts: {(~filt_counts).sum():,}")
print(f"  Fail min_genes:  {(~filt_genes).sum():,}")
print(f"  Fail max_MT%:    {(~filt_mt).sum():,}")

keep = filt_counts & filt_genes & filt_mt
adata_sp = adata_sp[keep].copy()

n_after = adata_sp.n_obs
n_removed = n_before - n_after
print(f"\n  Removed: {n_removed:,} spots ({n_removed/n_before*100:.1f}%)")
print(f"  After QC: {adata_sp.n_obs:,} spots × {adata_sp.n_vars:,} genes")

# ── Post-QC summary per group ───────────────────────────────────────────────
print(f"\n── Post-QC summary by diagnosis group ──\n")
print(f"  {'Group':<10s} {'n_spots':>8s} {'med_UMI':>10s} {'med_genes':>10s} {'med_MT%':>10s}")
print(f"  {'─'*52}")
for grp in ['Control', 'earlyAD', 'AD', 'AD_DS']:
    mask = adata_sp.obs['group'] == grp
    if mask.sum() == 0:
        continue
    print(f"  {grp:<10s} {mask.sum():>8d} "
          f"{adata_sp.obs.loc[mask, 'total_counts'].median():>10.0f} "
          f"{adata_sp.obs.loc[mask, 'n_genes_by_counts'].median():>10.0f} "
          f"{adata_sp.obs.loc[mask, 'pct_counts_mt'].median():>10.1f}")

# ── Post-QC per sample ──────────────────────────────────────────────────────
print(f"\n── Post-QC per sample ──\n")
print(f"  {'Sample':<35s} {'Group':<10s} {'n_spots':>8s} {'med_UMI':>10s}")
print(f"  {'─'*67}")
for sid in sorted(adata_sp.obs['sample_id'].unique()):
    mask = adata_sp.obs['sample_id'] == sid
    grp = adata_sp.obs.loc[mask, 'group'].iloc[0]
    print(f"  {sid:<35s} {grp:<10s} {mask.sum():>8d} "
          f"{adata_sp.obs.loc[mask, 'total_counts'].median():>10.0f}")

# Update samples list
samples = sorted(adata_sp.obs['sample_id'].unique())
n_samples = len(samples)

# ── Save QC checkpoint ───────────────────────────────────────────────────────
CHECKPOINT_QC = DATA_DIR / f'{DATASET}_qc.h5ad'
adata_sp.write(CHECKPOINT_QC)
file_size = CHECKPOINT_QC.stat().st_size / 1e9
print(f"\n✓ QC checkpoint saved: {CHECKPOINT_QC} ({file_size:.2f} GB)")
print(f"  {adata_sp.n_obs:,} spots × {adata_sp.n_vars:,} genes, {n_samples} samples")
print(f"\n→ Next: Cell 2 (Preprocess — normalize + regress UMI)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2: PREPROCESS — NORMALIZE + PEARSON RESIDUALS (SCTransform-style)
# ═══════════════════════════════════════════════════════════════════════════════
# Env: omicverse
# ═══════════════════════════════════════════════════════════════════════════════

from scipy.stats import spearmanr

print("=" * 60)
print("PREPROCESSING (Normalize + Pearson Residuals)")
print("=" * 60)

# Reload from QC checkpoint (fresh start)
CHECKPOINT_QC = DATA_DIR / f'{DATASET}_qc.h5ad'
adata_sp = sc.read_h5ad(CHECKPOINT_QC)
print(f"\n  Loaded QC checkpoint: {adata_sp.n_obs:,} spots × {adata_sp.n_vars:,} genes")
print(f"  X dtype: {adata_sp.X.dtype}, range: [{adata_sp.X.min()}, {adata_sp.X.max()}]")

# Step 1: Store raw counts
adata_sp.layers['raw_counts'] = adata_sp.X.copy()
print(f"\n  ✓ layers['raw_counts'] stored")

# Step 2: Log-normalize (for sc.tl.score_genes / module scoring later)
sc.pp.normalize_total(adata_sp, target_sum=1e4)
sc.pp.log1p(adata_sp)
adata_sp.layers['lognorm'] = adata_sp.X.copy()
print(f"  ✓ normalize_total + log1p → layers['lognorm']")
print(f"    range: [{adata_sp.X.min():.2f}, {adata_sp.X.max():.2f}]")

# Step 3: Pearson residuals (SCTransform-style, clip=30)
adata_sp.X = adata_sp.layers['raw_counts'].copy().astype(np.float64)
print(f"\n  Computing Pearson residuals (analytic, clip=30)...")
print(f"  X dtype after cast: {adata_sp.X.dtype}")

n_zero_spots = (np.array(adata_sp.X.sum(axis=1)).flatten() == 0).sum()
print(f"  Zero-count spots: {n_zero_spots}")

sc.experimental.pp.normalize_pearson_residuals(adata_sp, clip=30)
adata_sp.layers['pearson_residuals'] = adata_sp.X.copy()

pr_dense = np.array(adata_sp.X.todense() if hasattr(adata_sp.X, 'todense') else adata_sp.X)
n_nan = np.isnan(pr_dense).sum()
n_total = pr_dense.size
print(f"  NaN count: {n_nan:,} / {n_total:,} ({100*n_nan/n_total:.2f}%)")
print(f"  ✓ Pearson residuals → layers['pearson_residuals']")
print(f"    range: [{np.nanmin(pr_dense):.2f}, {np.nanmax(pr_dense):.2f}]")
del pr_dense

# Step 4: Verify UMI confounding reduced
print(f"\n── UMI correlation check (per group) ──\n")
print(f"  {'Group':<10s} {'n_spots':>8s} {'lognorm ρ':>12s} {'pearson ρ':>12s}")
print(f"  {'─'*46}")

for grp in ['Control', 'earlyAD', 'AD', 'AD_DS']:
    mask = (adata_sp.obs['group'] == grp).values
    if mask.sum() == 0:
        continue

    lognorm_vals = np.array(adata_sp.layers['lognorm'][mask].mean(axis=1)).flatten()
    rho_log, _ = spearmanr(lognorm_vals, adata_sp.obs.loc[mask, 'total_counts'].values)

    pr_sub = np.array(adata_sp.layers['pearson_residuals'][mask].todense()
                      if hasattr(adata_sp.layers['pearson_residuals'], 'todense')
                      else adata_sp.layers['pearson_residuals'][mask])
    pr_vals = np.nanmean(pr_sub, axis=1).flatten()
    rho_pr, _ = spearmanr(pr_vals, adata_sp.obs.loc[mask, 'total_counts'].values)

    print(f"  {grp:<10s} {mask.sum():>8d} {rho_log:>12.4f} {rho_pr:>12.4f}")

# ── Save checkpoint ──────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SAVING PREPROCESSED CHECKPOINT")
print("=" * 60)
print(f"\n  .X = pearson_residuals (UMI-corrected, for SenePy)")
print(f"  layers['raw_counts']        = raw integer counts")
print(f"  layers['lognorm']           = log-normalized (for sc.tl.score_genes)")
print(f"  layers['pearson_residuals'] = SCTransform-style residuals")
print(f"  Layers: {list(adata_sp.layers.keys())}")

CHECKPOINT_PREP = DATA_DIR / f'{DATASET}_preprocessed.h5ad'
adata_sp.write(CHECKPOINT_PREP)
file_size = CHECKPOINT_PREP.stat().st_size / 1e9
print(f"\n✓ Preprocessed checkpoint saved ({file_size:.2f} GB)")
print(f"  {adata_sp.n_obs:,} spots × {adata_sp.n_vars:,} genes")
print(f"\n→ Switch to senepy env for Cell 3 (SenePy scoring)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3: SENEPY SCORING — HIPPOCAMPUS HUB
# ═══════════════════════════════════════════════════════════════════════════════
# Env: senepy
# ═══════════════════════════════════════════════════════════════════════════════

import senepy as sp
import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
import scipy.sparse as sp_sparse
import warnings
warnings.filterwarnings('ignore')

BASE_DIR    = Path('/fs/scratch/PAS2598/senescence_analysis')
DATASET     = 'morabito_dsad'
DATA_DIR    = BASE_DIR / 'data' / '10_spatial'

SENEPY_TISSUE    = 'hippocampus'
CELL_TYPE_COLUMN = 'dominant_celltype'
SEX_COLUMN       = 'sex'

# ── Load preprocessed checkpoint ──────────────────────────────────────────────
CHECKPOINT_PREP = DATA_DIR / f'{DATASET}_preprocessed.h5ad'
adata_sp = sc.read_h5ad(CHECKPOINT_PREP)

print("=" * 60)
print("SENEPY SCORING")
print("=" * 60)
print(f"\n  Loaded: {adata_sp.n_obs:,} spots × {adata_sp.n_vars:,} genes")
print(f"  .X = pearson_residuals")
print(f"  Layers: {list(adata_sp.layers.keys())}")

print(f"\nScoring parameters:")
print(f"  Species: Human")
print(f"  Tissue: {SENEPY_TISSUE}")
print(f"  Cell type column: {CELL_TYPE_COLUMN}")
print(f"  Sex column: {SEX_COLUMN}")

# ── Remove zero-variance genes (NaN in Pearson residuals) ─────────────────────
print(f"\n0. Removing zero-variance genes...")
if sp_sparse.issparse(adata_sp.X):
    nan_per_gene = np.isnan(np.array(adata_sp.X.todense())).all(axis=0)
else:
    nan_per_gene = np.isnan(adata_sp.X).all(axis=0)

n_remove = nan_per_gene.sum()
print(f"   Genes with all-NaN residuals: {n_remove:,} / {adata_sp.n_vars:,}")

adata_sp = adata_sp[:, ~nan_per_gene].copy()
print(f"   ✓ Retained: {adata_sp.n_vars:,} genes")
print(f"   X range: [{np.nanmin(adata_sp.X):.2f}, {np.nanmax(adata_sp.X):.2f}]")

# Verify no NaN remaining
if sp_sparse.issparse(adata_sp.X):
    n_nan = np.isnan(np.array(adata_sp.X.todense())).sum()
else:
    n_nan = np.isnan(adata_sp.X).sum()
print(f"   Remaining NaN: {n_nan:,}")

# ── Step 1: Load SenePy hubs ─────────────────────────────────────────────────
print(f"\n1. Loading SenePy hubs...")
hubs = sp.load_hubs(species='Human')
print(f"   ✓ Loaded {len(hubs.metadata)} hub entries")

# ── Step 2: Filter for tissue ─────────────────────────────────────────────────
print(f"\n2. Filtering for {SENEPY_TISSUE} tissue...")
brain_hubs = hubs.metadata[hubs.metadata.tissue == SENEPY_TISSUE]
print(f"   ✓ Found {len(brain_hubs)} {SENEPY_TISSUE}-specific hubs")

# ── Step 3: Merge hubs ────────────────────────────────────────────────────────
print(f"\n3. Merging hubs...")
hubs.merge_hubs(brain_hubs, new_name='brain')
print(f"   ✓ Hubs merged")

# ── Step 4: Create translator ─────────────────────────────────────────────────
print(f"\n4. Creating gene translator...")
translator = sp.translator(hub=hubs.hubs, data=adata_sp)
print(f"   ✓ Translator created")

# ── Step 5: Score all spots ───────────────────────────────────────────────────
print(f"\n5. Scoring {adata_sp.n_obs:,} spots...")
print(f"   (identifiers: {CELL_TYPE_COLUMN} × {SEX_COLUMN})")

adata_sp.obs['sen_score'] = sp.score_all_cells(
    adata_sp,
    hubs.hubs['brain'],
    identifiers=[CELL_TYPE_COLUMN, SEX_COLUMN],
    translator=translator
)

print(f"\n✓ Senescence scoring complete")

scores = adata_sp.obs['sen_score']
print(f"\nScore statistics:")
print(f"  Mean:   {scores.mean():.4f}")
print(f"  Median: {scores.median():.4f}")
print(f"  Std:    {scores.std():.4f}")
print(f"  Range:  [{scores.min():.4f}, {scores.max():.4f}]")
print(f"  NaN:    {scores.isna().sum()}")

# ── Score by group ────────────────────────────────────────────────────────────
print(f"\n── Score by group ──\n")
print(f"  {'Group':<10s} {'n_spots':>8s} {'mean':>8s} {'median':>8s} {'std':>8s}")
print(f"  {'─'*46}")

for grp in ['Control', 'earlyAD', 'AD', 'AD_DS']:
    mask = adata_sp.obs['group'] == grp
    if mask.sum() == 0:
        continue
    s = scores[mask]
    print(f"  {grp:<10s} {mask.sum():>8d} {s.mean():>8.4f} {s.median():>8.4f} {s.std():>8.4f}")

# ── UMI correlation check ─────────────────────────────────────────────────────
print(f"\n── Senescence score vs UMI ──\n")
for grp in ['Control', 'earlyAD', 'AD', 'AD_DS']:
    mask = (adata_sp.obs['group'] == grp).values
    if mask.sum() == 0:
        continue
    rho, pval = spearmanr(scores[mask].values, adata_sp.obs.loc[mask, 'total_counts'].values)
    print(f"  {grp:<10s} ρ={rho:.4f}  p={pval:.2e}")

# ── Save scored checkpoint ────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SAVING SCORED CHECKPOINT")
print("=" * 60)

CHECKPOINT_SCORED = DATA_DIR / f'{DATASET}_senepy_scored.h5ad'
adata_sp.write(CHECKPOINT_SCORED)
file_size = CHECKPOINT_SCORED.stat().st_size / 1e9
print(f"\n✓ Saved ({file_size:.2f} GB)")
print(f"  {adata_sp.n_obs:,} spots × {adata_sp.n_vars:,} genes")
print(f"  New column: adata_sp.obs['sen_score']")
print(f"\n→ Next: Cell 4 (per sample labeling)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4: LABEL SENESCENT SPOTS — PER-SAMPLE THRESHOLD
# ═══════════════════════════════════════════════════════════════════════════════
# Env: senepy
# ═══════════════════════════════════════════════════════════════════════════════

SD_THRESHOLD = 2
RESULTS_DIR  = BASE_DIR / 'results' / '10_spatial' / DATASET
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("SENESCENT SPOT LABELING — PER-SAMPLE, PER-CELL-TYPE THRESHOLD")
print("=" * 60)
print(f"\nMethod: mean + {SD_THRESHOLD} × SD within each sample × cell type")

adata_sp.obs['sen_label'] = 'Non-SnC'
adata_sp.obs['is_senescent'] = 0

samples = adata_sp.obs['sample_id'].unique()
cell_types = sorted(adata_sp.obs['dominant_celltype'].unique())

summary_rows = []

for ct in cell_types:
    ct_mask = adata_sp.obs['dominant_celltype'] == ct

    print(f"\n  {ct}")
    print(f"  {'Sample':<20s} {'Group':<10s} {'Mean':>8s} {'SD':>8s} {'Thresh':>8s} {'SnC':>6s} {'Total':>6s} {'%SnC':>7s}")
    print(f"  {'─'*72}")

    for sid in samples:
        sample_ct_mask = ct_mask & (adata_sp.obs['sample_id'] == sid)
        scores = adata_sp.obs.loc[sample_ct_mask, 'sen_score']

        group = adata_sp.obs.loc[adata_sp.obs['sample_id'] == sid, 'group'].iloc[0]

        if len(scores) < 5:
            print(f"  {sid:<20s} {group:<10s} {'(skip: <5 spots)':>50s}")
            continue

        s_mean = scores.mean()
        s_std = scores.std()
        thresh = s_mean + SD_THRESHOLD * s_std

        snc_mask = sample_ct_mask & (adata_sp.obs['sen_score'] >= thresh)
        adata_sp.obs.loc[snc_mask, 'sen_label'] = 'SnC'
        adata_sp.obs.loc[snc_mask, 'is_senescent'] = 1

        n_snc = snc_mask.sum()
        n_tot = sample_ct_mask.sum()
        pct = n_snc / n_tot * 100 if n_tot > 0 else 0

        print(f"  {sid:<20s} {group:<10s} {s_mean:>8.4f} {s_std:>8.4f} {thresh:>8.4f} "
              f"{n_snc:>6d} {n_tot:>6d} {pct:>6.1f}%")

        summary_rows.append({
            'cell_type': ct, 'sample_id': sid, 'group': group,
            'mean': s_mean, 'sd': s_std, 'threshold': thresh,
            'n_snc': n_snc, 'n_total': n_tot, 'pct_snc': pct
        })

# ── Sample totals ─────────────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"  {'Sample':<20s} {'Group':<10s} {'SnC':>6s} {'Total':>6s} {'%SnC':>7s}")
print(f"  {'─'*52}")

for sid in samples:
    mask = adata_sp.obs['sample_id'] == sid
    group = adata_sp.obs.loc[mask, 'group'].iloc[0]
    n_snc = (adata_sp.obs.loc[mask, 'is_senescent'] == 1).sum()
    total = mask.sum()
    print(f"  {sid:<20s} {group:<10s} {n_snc:>6d} {total:>6d} {n_snc/total*100:>6.1f}%")

# ── Group totals ──────────────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"  {'Group':<10s} {'SnC':>6s} {'Total':>6s} {'%SnC':>7s}")
print(f"  {'─'*32}")

for grp in ['Control', 'earlyAD', 'AD', 'AD_DS']:
    mask = adata_sp.obs['group'] == grp
    if mask.sum() == 0:
        continue
    n_snc = (adata_sp.obs.loc[mask, 'is_senescent'] == 1).sum()
    total = mask.sum()
    print(f"  {grp:<10s} {n_snc:>6d} {total:>6d} {n_snc/total*100:>6.1f}%")

# ── Save ──────────────────────────────────────────────────────────────────────
df_thresh = pd.DataFrame(summary_rows)
df_thresh.to_csv(RESULTS_DIR / 'senescence_thresholds_per_sample.csv', index=False)
print(f"\n✓ Thresholds saved: {RESULTS_DIR / 'senescence_thresholds_per_sample.csv'}")

CHECKPOINT_LABELED = DATA_DIR / f'{DATASET}_senepy_labeled.h5ad'
adata_sp.write(CHECKPOINT_LABELED)
file_size = CHECKPOINT_LABELED.stat().st_size / 1e9
print(f"✓ Labeled checkpoint saved ({file_size:.2f} GB)")
print(f"  New columns: sen_label, is_senescent")
print(f"\n→ Next: Cell 5 (Visualization)")

In [ ]:
print(f"  {'Sample':<35s} {'Group':<10s} {'Sex':>3s} {'Age':>4s} {'Spots':>6s} {'UMI_med':>10s} {'Genes_med':>10s}")
print(f"  {'─'*82}")

for sid in sorted(adata_sp.obs['sample_id'].unique()):
    mask = adata_sp.obs['sample_id'] == sid
    grp = adata_sp.obs.loc[mask, 'group'].iloc[0]
    sex = adata_sp.obs.loc[mask, 'sex'].iloc[0]
    age = adata_sp.obs.loc[mask, 'age'].iloc[0]
    umi_med = adata_sp.obs.loc[mask, 'total_counts'].median()
    genes_med = adata_sp.obs.loc[mask, 'n_genes_by_counts'].median()
    n_spots = mask.sum()
    
    if grp in ['Control', 'AD']:
        print(f"  {sid:<35s} {grp:<10s} {sex:>3s} {age:>4.0f} {n_spots:>6d} {umi_med:>10.0f} {genes_med:>10.0f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5: SUBSET SELECTED SAMPLES FOR VISUALIZATION
# ═══════════════════════════════════════════════════════════════════════════════
# Env: senepy
# ═══════════════════════════════════════════════════════════════════════════════

SELECTED = {
    'Dec_20_2021_Human5':        {'name': 'Control-F', 'group': 'Control', 'sex': 'F', 'age': 64},
    'Nov_24_2021_VisiumHuman_9': {'name': 'Control-M', 'group': 'Control', 'sex': 'M', 'age': 83},
    'Dec_13_2021_Human7':        {'name': 'AD-F',      'group': 'AD',      'sex': 'F', 'age': 90},
    'Nov_24_2021_VisiumHuman_7': {'name': 'AD-M',      'group': 'AD',      'sex': 'M', 'age': 90},
}

print("=" * 60)
print("SUBSETTING SELECTED SAMPLES")
print("=" * 60)

adata_sel = adata_sp[adata_sp.obs['sample_id'].isin(SELECTED.keys())].copy()

print(f"\n  Full dataset: {adata_sp.n_obs:,} spots")
print(f"  Selected:     {adata_sel.n_obs:,} spots")

print(f"\n  {'Sample':<30s} {'Name':<12s} {'Group':<10s} {'Sex':>3s} {'Age':>4s} {'Spots':>6s} {'%SnC':>6s}")
print(f"  {'─'*74}")

for sid, meta in SELECTED.items():
    mask = adata_sel.obs['sample_id'] == sid
    n_spots = mask.sum()
    n_snc = (adata_sel.obs.loc[mask, 'is_senescent'] == 1).sum()
    pct = n_snc / n_spots * 100 if n_spots > 0 else 0
    print(f"  {sid:<30s} {meta['name']:<12s} {meta['group']:<10s} {meta['sex']:>3s} {meta['age']:>4d} {n_spots:>6d} {pct:>5.1f}%")

print(f"\n✓ adata_sel ready for visualization")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6: MODULE SCORING — SENESCENCE HALLMARK GENE LISTS
# ═══════════════════════════════════════════════════════════════════════════════
# Env: omicverse
# Input: morabito_dsad_senepy_labeled.h5ad (from Cell 4)
# ═══════════════════════════════════════════════════════════════════════════════

import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR    = Path('/fs/scratch/PAS2598/senescence_analysis')
DATASET     = 'morabito_dsad'
DATA_DIR    = BASE_DIR / 'data' / '10_spatial'
FIGURES_DIR = BASE_DIR / 'figures' / '10_spatial' / DATASET
RESULTS_DIR = BASE_DIR / 'results' / '10_spatial' / DATASET
MARKERS_DIR = Path('/fs/ess/PDE0075/senescence/human/markers')

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Selected samples ─────────────────────────────────────────────────────────
SELECTED = {
    'Nov_24_2021_VisiumHuman_5':     {'name': 'F-79',  'group': 'Control', 'sex': 'F', 'age': 79},
    'Oct_2021_5':                    {'name': 'M-79',  'group': 'Control', 'sex': 'M', 'age': 79},
    'Nov_24_2021_VisiumHuman_13':    {'name': 'F-90',  'group': 'Control', 'sex': 'F', 'age': 90},
    'Nov_24_2021_VisiumHuman_1':     {'name': 'M-90',  'group': 'Control', 'sex': 'M', 'age': 90},
}

samples   = list(SELECTED.keys())
n_samples = len(samples)

def get_meta(sid):
    return SELECTED[sid]['name'], SELECTED[sid]['group']

# ── Load labeled data & subset ───────────────────────────────────────────────
print("=" * 60)
print("MODULE SCORING — SENESCENCE HALLMARK GENE LISTS")
print("=" * 60)

adata_sp = sc.read_h5ad(DATA_DIR / 'morabito_dsad_senepy_labeled.h5ad')
print(f"\n  Full dataset: {adata_sp.n_obs:,} spots × {adata_sp.n_vars:,} genes")

adata_sel = adata_sp[adata_sp.obs['sample_id'].isin(SELECTED.keys())].copy()
del adata_sp  # free memory

print(f"  Selected:     {adata_sel.n_obs:,} spots × {adata_sel.n_vars:,} genes")
print(f"  SnC: {adata_sel.obs['is_senescent'].sum()} / {adata_sel.n_obs}")
print(f"  Layers: {list(adata_sel.layers.keys())}")

# ── Switch to log-normalized layer ───────────────────────────────────────────
print(f"\n  Current .X range: [{adata_sel.X.min():.2f}, {adata_sel.X.max():.2f}]")

if 'lognorm' in adata_sel.layers:
    adata_sel.X = adata_sel.layers['lognorm'].copy()
    print(f"  ✓ Switched to 'lognorm' layer: [{adata_sel.X.min():.2f}, {adata_sel.X.max():.2f}]")
elif 'log1p' in adata_sel.layers:
    adata_sel.X = adata_sel.layers['log1p'].copy()
    print(f"  ✓ Switched to 'log1p' layer: [{adata_sel.X.min():.2f}, {adata_sel.X.max():.2f}]")
else:
    print(f"  Available layers: {list(adata_sel.layers.keys())}")
    print(f"  ⚠ No log-normalized layer found — check layer names")

# ── Load gene lists from Sloan et al. Table S9 ──────────────────────────────
print(f"\n{'='*60}")
print("LOADING SENESCENCE GENE LISTS")
print(f"{'='*60}")

sloan_raw = pd.read_excel(
    MARKERS_DIR / '1-s2.0-S2666979X25003830-mmc10.xlsx',
    sheet_name='Sen Gene Lists'
)

hallmark_names = [
    'p53_Targets', 'CellCycleArrest', 'SASP', 'AntiApoptosis',
    'DDR', 'CellSurfaceMarkers', 'LysosomalContent',
    'SD_TMC', 'SenMayo', 'Fridman_Up',
]
list_types = ['Individual hallmark'] * 7 + ['Multi-hallmark'] * 3

gene_lists = {}
for i, name in enumerate(hallmark_names):
    genes_raw = sloan_raw.iloc[1:, i].dropna().astype(str).str.strip()
    genes_raw = genes_raw[genes_raw != ''].tolist()
    detected = [g for g in genes_raw if g in adata_sel.var_names]
    gene_lists[name] = detected
    print(f"  {name:<20s}: {len(detected):>3d} / {len(genes_raw):<3d} detected "
          f"({len(detected)/max(len(genes_raw),1)*100:.0f}%)  [{list_types[i]}]")

# ── Score modules ────────────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("Scoring modules (sc.tl.score_genes)...")
print(f"{'─'*60}")

for name, genes in gene_lists.items():
    if len(genes) < 5:
        print(f"  {name:<20s}: SKIP (<5 detected genes)")
        continue
    sc.tl.score_genes(adata_sel, gene_list=genes, score_name=f'Score_{name}',
                      ctrl_size=min(100, len(genes)))
    vals = adata_sel.obs[f'Score_{name}']
    print(f"  {name:<20s}: mean={vals.mean():.4f}, sd={vals.std():.4f}")

score_cols = [c for c in adata_sel.obs.columns if c.startswith('Score_')]

# ── Mean scores: SnC vs Non-SnC per sample ───────────────────────────────────
print(f"\n{'='*60}")
print("MEAN MODULE SCORES: SnC vs Non-SnC")
print(f"{'='*60}")

for sid in samples:
    name, group = get_meta(sid)
    mask = adata_sel.obs['sample_id'] == sid
    snc_mask = mask & (adata_sel.obs['is_senescent'] == 1)
    non_mask = mask & (adata_sel.obs['is_senescent'] == 0)

    print(f"\n  {name} ({group}) — {snc_mask.sum()} SnC / {non_mask.sum()} Non-SnC")
    print(f"  {'List':<20s} {'SnC':>10s} {'Non-SnC':>10s} {'Diff':>10s}")
    print(f"  {'─'*53}")

    for sc_col in score_cols:
        nm = sc_col.replace('Score_', '')
        snc_mean = adata_sel.obs.loc[snc_mask, sc_col].mean()
        non_mean = adata_sel.obs.loc[non_mask, sc_col].mean()
        diff = snc_mean - non_mean
        print(f"  {nm:<20s} {snc_mean:>10.4f} {non_mean:>10.4f} {diff:>10.4f}")

# ── Mean scores by dominant cell type per sample ─────────────────────────────
print(f"\n{'='*60}")
print("MEAN MODULE SCORES BY DOMINANT CELL TYPE")
print(f"{'='*60}")

for sid in samples:
    name, group = get_meta(sid)
    mask = adata_sel.obs['sample_id'] == sid
    cts = sorted(adata_sel.obs.loc[mask, 'dominant_celltype'].unique())

    print(f"\n  {name} ({group})")
    header = f"  {'Cell Type':<16s}"
    for sc_col in score_cols:
        nm = sc_col.replace('Score_', '')
        header += f" {nm[:12]:>12s}"
    print(header)
    print(f"  {'─'*(16 + 12*len(score_cols) + len(score_cols))}")

    for ct in cts:
        ct_mask = mask & (adata_sel.obs['dominant_celltype'] == ct)
        n = ct_mask.sum()
        row = f"  {ct:<16s}"
        for sc_col in score_cols:
            val = adata_sel.obs.loc[ct_mask, sc_col].mean()
            row += f" {val:>12.4f}"
        row += f"  (n={n})"
        print(row)

# ── Save summary CSV ─────────────────────────────────────────────────────────
summary_rows = []
for sid in samples:
    name, group = get_meta(sid)
    mask = adata_sel.obs['sample_id'] == sid
    for status, s_mask in [('SnC', mask & (adata_sel.obs['is_senescent'] == 1)),
                           ('Non-SnC', mask & (adata_sel.obs['is_senescent'] == 0))]:
        row = {'sample': name, 'group': group, 'status': status, 'n_spots': s_mask.sum()}
        for sc_col in score_cols:
            row[sc_col.replace('Score_', '')] = adata_sel.obs.loc[s_mask, sc_col].mean()
        summary_rows.append(row)

pd.DataFrame(summary_rows).to_csv(RESULTS_DIR / 'module_scores_summary.csv', index=False)
print(f"\n✓ Saved: {RESULTS_DIR / 'module_scores_summary.csv'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7: EXPLORATORY VISUALIZATION — C2L & DOMINANT CELL TYPE
# ═══════════════════════════════════════════════════════════════════════════════
# Env: omicverse (continues from Cell 6)
# Requires: adata_sel, SELECTED, samples, n_samples, get_meta, score_cols
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ── 1. C2L cell type abundance spatial maps ──────────────────────────────────
plot_cts = ['Astrocyte', 'Oligodendrocyte', 'OPC', 'Microglia',
            'Excitatory', 'Inhibitory']

fig, axes = plt.subplots(n_samples, len(plot_cts), figsize=(4 * len(plot_cts), 4 * n_samples))
if n_samples == 1:
    axes = axes[np.newaxis, :]
fig.suptitle('Cell Type Abundance (cell2location)', fontsize=14, fontweight='bold')

for i, sid in enumerate(samples):
    name, group = get_meta(sid)
    mask = (adata_sel.obs['sample_id'] == sid).values
    coords = adata_sel[mask].obsm['spatial']
    for j, ct in enumerate(plot_cts):
        vals = adata_sel.obs.loc[mask, f'c2l_{ct}'].values
        im = axes[i, j].scatter(coords[:, 0], coords[:, 1],
                                c=vals, cmap='magma', s=4, edgecolors='none')
        axes[i, j].set_title(f"{name}: {ct}", fontsize=9)
        axes[i, j].invert_yaxis()
        axes[i, j].set_aspect('equal')
        axes[i, j].axis('off')
        plt.colorbar(im, ax=axes[i, j], shrink=0.7)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'c2l_spatial_celltypes.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print("✓ c2l_spatial_celltypes.pdf")

# ── 2. Dominant cell type map ─────────────────────────────────────────────────
all_cts = sorted(adata_sel.obs['dominant_celltype'].unique())
cmap_ct = plt.cm.get_cmap('tab20', len(all_cts))
ct_colors_tab = {ct: cmap_ct(i) for i, ct in enumerate(all_cts)}

fig, axes = plt.subplots(1, n_samples, figsize=(7 * n_samples, 6))
if n_samples == 1:
    axes = [axes]
fig.suptitle('Dominant Cell Type per Spot', fontsize=14, fontweight='bold')

for i, sid in enumerate(samples):
    name, group = get_meta(sid)
    mask = (adata_sel.obs['sample_id'] == sid).values
    coords = adata_sel[mask].obsm['spatial']
    colors = [ct_colors_tab[c] for c in adata_sel.obs.loc[mask, 'dominant_celltype']]
    axes[i].scatter(coords[:, 0], coords[:, 1], c=colors, s=4, edgecolors='none')
    axes[i].set_title(f"{name} ({group})")
    axes[i].invert_yaxis()
    axes[i].set_aspect('equal')
    axes[i].axis('off')

handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=ct_colors_tab[ct],
                   markersize=8, label=ct) for ct in all_cts]
fig.legend(handles=handles, loc='center right', bbox_to_anchor=(1.15, 0.5), fontsize=8)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'c2l_dominant_celltype.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print("✓ c2l_dominant_celltype.pdf")

# ── 3. Mean abundance per cell type per sample (bar plot) ─────────────────────
c2l_cols = [c for c in adata_sel.obs.columns if c.startswith('c2l_') and not c.startswith('c2l_fraction')]
cell_types = sorted([c.replace('c2l_', '') for c in c2l_cols])

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(cell_types))
width = 0.8 / n_samples

for i, sid in enumerate(samples):
    name, group = get_meta(sid)
    mask = adata_sel.obs['sample_id'] == sid
    means = [adata_sel.obs.loc[mask, f'c2l_{ct}'].mean() for ct in cell_types]
    ax.bar(x + i * width, means, width, label=f"{name} ({group})")

ax.set_xticks(x + width * (n_samples - 1) / 2)
ax.set_xticklabels(cell_types, rotation=45, ha='right')
ax.set_ylabel('Mean c2l Abundance')
ax.set_title('Mean Cell Type Abundance per Sample', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'c2l_mean_abundance_barplot.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print("✓ c2l_mean_abundance_barplot.pdf")

# ── 4. Cell type proportion per sample (stacked bar) ─────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

bottom = np.zeros(n_samples)
sample_labels = []

for ct in cell_types:
    fracs = []
    for idx, sid in enumerate(samples):
        name, group = get_meta(sid)
        if ct == cell_types[0]:
            sample_labels.append(f"{name}\n({group})")
        mask = adata_sel.obs['sample_id'] == sid
        total = sum(adata_sel.obs.loc[mask, f'c2l_{c}'].mean() for c in cell_types)
        fracs.append(adata_sel.obs.loc[mask, f'c2l_{ct}'].mean() / total)
    ax.bar(range(n_samples), fracs, bottom=bottom, label=ct,
           color=ct_colors_tab.get(ct, None))
    bottom += fracs

ax.set_xticks(range(n_samples))
ax.set_xticklabels(sample_labels[:n_samples])
ax.set_ylabel('Proportion')
ax.set_title('Cell Type Proportions per Sample', fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'c2l_proportion_stacked.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print("✓ c2l_proportion_stacked.pdf")

# ── 5. Dominant cell type counts per sample ───────────────────────────────────
fig, axes = plt.subplots(1, n_samples, figsize=(6 * n_samples, 5))
if n_samples == 1:
    axes = [axes]

for i, sid in enumerate(samples):
    name, group = get_meta(sid)
    mask = adata_sel.obs['sample_id'] == sid
    counts = adata_sel.obs.loc[mask, 'dominant_celltype'].value_counts()
    colors_bar = [ct_colors_tab[ct] for ct in counts.index]
    axes[i].barh(counts.index, counts.values, color=colors_bar)
    axes[i].set_xlabel('Number of Spots')
    axes[i].set_title(f"{name} ({group})\nn={mask.sum()} spots", fontweight='bold')
    for j, (ct, val) in enumerate(counts.items()):
        axes[i].text(val + 10, j, f"{val} ({val/mask.sum()*100:.1f}%)", va='center', fontsize=7)

plt.suptitle('Dominant Cell Type Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'c2l_dominant_counts.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print("✓ c2l_dominant_counts.pdf")

# ── 6. C2L extraction summary table ──────────────────────────────────────────
print("\n" + "=" * 60)
print("C2L EXTRACTION SUMMARY")
print("=" * 60)

for sid in samples:
    name, group = get_meta(sid)
    mask = adata_sel.obs['sample_id'] == sid
    n = mask.sum()

    print(f"\n  {name} ({group}, n={n} spots)")
    print(f"  {'Cell Type':<20s} {'Mean Abund':>12s} {'Mean Frac':>10s} {'Dominant':>10s}")
    print(f"  {'─'*56}")

    total_abund = sum(adata_sel.obs.loc[mask, f'c2l_{ct}'].mean() for ct in cell_types)
    dom_counts = adata_sel.obs.loc[mask, 'dominant_celltype'].value_counts()

    for ct in cell_types:
        mean_a = adata_sel.obs.loc[mask, f'c2l_{ct}'].mean()
        frac = mean_a / total_abund
        n_dom = dom_counts.get(ct, 0)
        pct_dom = n_dom / n * 100
        print(f"  {ct:<20s} {mean_a:>12.2f} {frac:>10.3f} {n_dom:>5d} ({pct_dom:4.1f}%)")

print(f"\n✓ All exploratory figures saved to {FIGURES_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9A: COMPACT DOMINANT CELL TYPE MAP (PUBLICATION QUALITY)
# ═══════════════════════════════════════════════════════════════════════════════
# Env: omicverse (continues from Cell 8)
# Requires: adata_sel, SELECTED, samples, n_samples, get_meta, FIGURES_DIR
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.gridspec as gridspec
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'pdf.fonttype': 42,
})

ct_colors = {
    'Excitatory':      '#2ca02c',
    'Inhibitory':      '#ff7f0e',
    'Oligodendrocyte': '#9467bd',
    'Astrocyte':       '#1f77b4',
    'Microglia':       '#8c564b',
    'OPC':             '#e377c2',
    'Endothelial':     '#d62728',
    'Pericyte':        '#17becf',
    'PVM':             '#bcbd22',
    'VLMC':            '#f7b6d2',
    'VSMC':            '#7f7f7f',
    'Adaptive':        '#aec7e8',
}

present_cts = sorted(adata_sel.obs['dominant_celltype'].unique())

fig = plt.figure(figsize=(7, 3.5 * n_samples + 0.5))
gs = gridspec.GridSpec(n_samples, 2, width_ratios=[1, 0.15], wspace=0.02)

for i, sid in enumerate(samples):
    name, group = get_meta(sid)
    mask = (adata_sel.obs['sample_id'] == sid).values
    coords = adata_sel[mask].obsm['spatial']
    dom = adata_sel.obs.loc[mask, 'dominant_celltype'].values

    ax = fig.add_subplot(gs[i, 0])
    for ct in present_cts:
        ct_mask = dom == ct
        if ct_mask.sum() > 0:
            ax.scatter(coords[ct_mask, 0], coords[ct_mask, 1],
                       c=ct_colors.get(ct, '#999'), s=3, alpha=0.85,
                       edgecolors='none', rasterized=True)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"{name} ({group})", fontsize=9, fontweight='bold', pad=4)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.5)
        spine.set_edgecolor('#333333')

legend_ax = fig.add_subplot(gs[:, 1])
legend_ax.axis('off')
handles = [Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=ct_colors.get(ct, '#999'),
                   markersize=5, markeredgewidth=0, label=ct)
           for ct in present_cts]
legend_ax.legend(handles=handles, loc='center left', frameon=False,
                 fontsize=6.5, labelspacing=0.8, handletextpad=0.3,
                 borderpad=0)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'dominant_celltype_compact.pdf'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'dominant_celltype_compact.svg'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ dominant_celltype_compact.pdf / .svg")


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9B: PUBLICATION MULTI-PANEL — CELL TYPES + MODULE SCORES + MODULE SCORES w/ SnC
# ═══════════════════════════════════════════════════════════════════════════════

from matplotlib.colors import LinearSegmentedColormap

SPOT_SIZE = 10
SNC_SIZE = 25

# ── Custom colormaps ──────────────────────────────────────────────────────────
cmap_defs = {
    'SASP':             ['#FFFFFF', '#FFF7BC', '#FEC44F', '#EC7014', '#CC4C02', '#8C2D04'],
    'p53_Targets':      ['#FFFFFF', '#EFEDF5', '#BCBDDC', '#807DBA', '#6A51A3', '#4A1486'],
    'DDR':              ['#FFFFFF', '#E5F5E0', '#A1D99B', '#41AB5D', '#238B45', '#005A32'],
    'LysosomalContent': ['#FFFFFF', '#FEE0D2', '#FC9272', '#EF3B2C', '#CB181D', '#99000D'],
    'SenMayo':          ['#FFFFFF', '#DEEBF7', '#9ECAE1', '#4292C6', '#2171B5', '#084594'],
    'Fridman_Up':       ['#FFFFFF', '#FFF5EB', '#FDD0A2', '#FD8D3C', '#D94801', '#8C2D04'],
}

score_rows = ['SASP', 'p53_Targets', 'DDR', 'LysosomalContent', 'SenMayo', 'Fridman_Up']
score_cols_plot = [f'Score_{s}' for s in score_rows if f'Score_{s}' in adata_sel.obs.columns]
score_names_plot = [s.replace('Score_', '') for s in score_cols_plot]

cmaps = {}
for sn in score_names_plot:
    if sn in cmap_defs:
        cmaps[sn] = LinearSegmentedColormap.from_list(sn, cmap_defs[sn], N=256)
    else:
        cmaps[sn] = 'RdYlBu_r'

# Layout: Row 0 = Cell Type
#         Then alternating pairs: score clean, score + SnC
#         e.g. SASP, SASP+SnC, p53, p53+SnC, ...
n_scores = len(score_names_plot)
n_rows = 1 + n_scores * 2
row_titles = ['Cell Type']
for sn in score_names_plot:
    row_titles.append(sn)
    row_titles.append(f'{sn} + SnC')

def plot_snc_overlay(ax, snc_x, snc_y):
    ax.scatter(snc_x, snc_y, c='white', s=SNC_SIZE + 15, alpha=1,
               zorder=4, rasterized=True)
    ax.scatter(snc_x, snc_y, c='#d62728', s=SNC_SIZE, alpha=0.95,
               edgecolors='black', linewidths=0.5, zorder=5, rasterized=True)

def format_ax(ax):
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)
        spine.set_edgecolor('#333333')

# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(8, 2.8 * n_rows + 0.5))
gs = gridspec.GridSpec(n_rows, n_samples + 1,
                       width_ratios=[1] * n_samples + [0.15],
                       hspace=0.18, wspace=0.08,
                       left=0.1, right=0.95, top=0.96, bottom=0.03)

for row_idx in range(n_rows):
    for col_idx, sid in enumerate(samples):
        ax = fig.add_subplot(gs[row_idx, col_idx])
        name, group = get_meta(sid)

        mask = (adata_sel.obs['sample_id'] == sid).values
        coords_raw = adata_sel[mask].obsm['spatial']
        valid = ~np.isnan(coords_raw).any(axis=1)
        coords = coords_raw[valid]
        x, y = coords[:, 0], coords[:, 1]

        if row_idx == 0:
            # Cell type map
            dom = adata_sel.obs.loc[mask, 'dominant_celltype'].values[valid]
            for ct in present_cts:
                ct_mask = dom == ct
                if ct_mask.sum() > 0:
                    ax.scatter(x[ct_mask], y[ct_mask], c=ct_colors.get(ct, '#999'),
                               s=SPOT_SIZE, alpha=0.85, edgecolors='none', rasterized=True)
        else:
            # Alternating pairs: odd = clean, even = +SnC
            pair_idx = (row_idx - 1) // 2   # which score (0..n_scores-1)
            is_snc_row = (row_idx - 1) % 2 == 1  # odd offset = +SnC

            sc_name = score_names_plot[pair_idx]
            sc_col = score_cols_plot[pair_idx]
            vals = adata_sel.obs.loc[mask, sc_col].values[valid]
            vmin = np.nanpercentile(vals, 2)
            vmax = np.nanpercentile(vals, 98)
            ax.scatter(x, y, c=vals, cmap=cmaps[sc_name], s=SPOT_SIZE, alpha=0.85,
                       vmin=vmin, vmax=vmax, edgecolors='none', rasterized=True)

            if is_snc_row:
                is_snc = adata_sel.obs.loc[mask, 'is_senescent'].values[valid] == 1
                plot_snc_overlay(ax, x[is_snc], y[is_snc])
                n_snc = is_snc.sum()
                ax.text(0.5, -0.02, f'n={n_snc}', transform=ax.transAxes,
                        ha='center', fontsize=5.5, color='#666666', style='italic')

        format_ax(ax)

        if row_idx == 0:
            ax.set_title(f"{name} ({group})", fontsize=9, fontweight='bold', pad=4)

        if col_idx == 0:
            ax.set_ylabel(row_titles[row_idx], fontsize=7, fontweight='bold',
                          rotation=90, labelpad=8)

    # ── Legend column ─────────────────────────────────────────────────────────
    legend_ax = fig.add_subplot(gs[row_idx, n_samples])
    legend_ax.axis('off')

    if row_idx == 0:
        handles = [Line2D([0], [0], marker='o', color='w',
                          markerfacecolor=ct_colors.get(ct, '#999'),
                          markersize=4, markeredgewidth=0, label=ct)
                   for ct in present_cts]
        legend_ax.legend(handles=handles, loc='center left', frameon=False,
                         fontsize=5, labelspacing=0.5, handletextpad=0.2, borderpad=0)
    else:
        pair_idx = (row_idx - 1) // 2
        is_snc_row = (row_idx - 1) % 2 == 1
        # Only add colorbar on the clean row (avoid duplicate)
        if not is_snc_row:
            sc_name = score_names_plot[pair_idx]
            sc_col = score_cols_plot[pair_idx]
            vals_all = adata_sel.obs[sc_col].dropna().values
            vmin = np.nanpercentile(vals_all, 2)
            vmax = np.nanpercentile(vals_all, 98)
            sm = plt.cm.ScalarMappable(cmap=cmaps[sc_name], norm=plt.Normalize(vmin, vmax))
            cbar = plt.colorbar(sm, ax=legend_ax, fraction=0.8, aspect=8)
            cbar.ax.tick_params(labelsize=5)
            cbar.set_label(sc_name, fontsize=6)

# ── SnC legend at bottom ─────────────────────────────────────────────────────
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728',
           markeredgecolor='black', markeredgewidth=0.5, markersize=6,
           label='Senescent spot (SnC)')
]
fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 0.001),
           frameon=True, fancybox=True, fontsize=7)

plt.savefig(str(FIGURES_DIR / 'Fig_multipanel_celltype_modules.pdf'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_multipanel_celltype_modules.svg'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_multipanel_celltype_modules.pdf / .svg")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10: SnC SPATIAL CLUSTERING & DENSITY ANALYSIS (Publication Quality)
# ═══════════════════════════════════════════════════════════════════════════════
# Env: omicverse
# Input: morabito_dsad_senepy_labeled.h5ad (from Cell 4)
# ═══════════════════════════════════════════════════════════════════════════════

import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial import KDTree
from scipy.stats import gaussian_kde
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
import warnings
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR    = Path('/fs/scratch/PAS2598/senescence_analysis')
DATASET     = 'morabito_dsad_control'
DATA_DIR    = BASE_DIR / 'data' / '10_spatial'
FIGURES_DIR = BASE_DIR / 'figures' / '10_spatial' / DATASET
RESULTS_DIR = BASE_DIR / 'results' / '10_spatial' / DATASET

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Global rcParams ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 7,
    'axes.titlesize': 8,
    'axes.labelsize': 7,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 5.5,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'pdf.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.4,
    'ytick.major.width': 0.4,
    'xtick.major.size': 2,
    'ytick.major.size': 2,
})

# ── Selected samples ─────────────────────────────────────────────────────────
SELECTED = {
    'Nov_24_2021_VisiumHuman_5':     {'name': 'F-79',  'group': 'Control', 'sex': 'F', 'age': 79},
    'Oct_2021_5':                    {'name': 'M-79',  'group': 'Control', 'sex': 'M', 'age': 79},
    'Nov_24_2021_VisiumHuman_13':    {'name': 'F-90',  'group': 'Control', 'sex': 'F', 'age': 90},
    'Nov_24_2021_VisiumHuman_1':     {'name': 'M-90',  'group': 'Control', 'sex': 'M', 'age': 90},
}

samples   = list(SELECTED.keys())
n_samples = len(samples)

# Sample colors (consistent across all plots)
SAMPLE_COLORS = {'F-79': '#4477AA', 'M-79': '#EE6677', 'F-90': '#228833', 'M-90': '#CCBB44'}

def get_meta(sid):
    return SELECTED[sid]['name'], SELECTED[sid]['group']

def format_spatial_ax(ax):
    """Clean spatial axis — no ticks, thin border."""
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)
        spine.set_edgecolor('#444444')

# ── Precompute per-sample data ───────────────────────────────────────────────
sample_data = {}
for sid in samples:
    name, _ = get_meta(sid)
    mask = (adata_sel.obs['sample_id'] == sid).values
    coords = adata_sel[mask].obsm['spatial']
    is_snc = (adata_sel.obs.loc[mask, 'is_senescent'] == 1).values
    sample_data[sid] = {
        'name': name, 'coords': coords, 'is_snc': is_snc,
        'snc_coords': coords[is_snc], 'non_coords': coords[~is_snc],
    }


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: COMBINED PANEL — SnC MAP + KDE DENSITY (2 rows × 4 cols)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print("FIGURE 1: SnC Map + KDE Density")
print(f"{'='*60}")

cmap_density = LinearSegmentedColormap.from_list(
    'snc_heat', ['#FFFFFF', '#FEE5D9', '#FCAE91', '#FB6A4A', '#CB181D', '#67000D'], N=256
)

fig = plt.figure(figsize=(7.2, 3.8))
gs = gridspec.GridSpec(2, n_samples, hspace=0.15, wspace=0.06,
                       left=0.02, right=0.98, top=0.90, bottom=0.04)

for col, sid in enumerate(samples):
    d = sample_data[sid]

    # ── Row 0: Spatial map with SnC overlay ──────────────────────────────────
    ax = fig.add_subplot(gs[0, col])
    ax.scatter(d['coords'][:, 0], d['coords'][:, 1], c='#E8E8E8', s=1.5,
               alpha=0.4, edgecolors='none', rasterized=True)
    ax.scatter(d['snc_coords'][:, 0], d['snc_coords'][:, 1], c='#D62728',
               s=8, alpha=0.9, edgecolors='black', linewidths=0.2, rasterized=True, zorder=3)
    format_spatial_ax(ax)
    n_snc = d['is_snc'].sum()
    n_tot = len(d['is_snc'])
    ax.set_title(f"{d['name']}\n{n_snc} SnC / {n_tot} ({n_snc/n_tot*100:.1f}%)",
                 fontsize=7, fontweight='bold', pad=3)
    if col == 0:
        ax.set_ylabel('SnC Map', fontsize=7, fontweight='bold', labelpad=4)

    # ── Row 1: KDE density ───────────────────────────────────────────────────
    ax = fig.add_subplot(gs[1, col])
    ax.scatter(d['coords'][:, 0], d['coords'][:, 1], c='#F5F5F5', s=1,
               alpha=0.3, edgecolors='none', rasterized=True)

    if n_snc >= 5:
        try:
            kde = gaussian_kde(d['snc_coords'].T, bw_method=0.15)
            x_min, x_max = d['coords'][:, 0].min(), d['coords'][:, 0].max()
            y_min, y_max = d['coords'][:, 1].min(), d['coords'][:, 1].max()
            pad = 0.05 * max(x_max - x_min, y_max - y_min)
            xx, yy = np.mgrid[x_min-pad:x_max+pad:200j, y_min-pad:y_max+pad:200j]
            zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
            ax.contourf(xx, yy, zz, levels=12, cmap=cmap_density, alpha=0.85)
            ax.contour(xx, yy, zz, levels=4, colors='#555555', linewidths=0.25, alpha=0.5)
            ax.scatter(d['snc_coords'][:, 0], d['snc_coords'][:, 1], c='#D62728',
                       s=4, alpha=0.7, edgecolors='black', linewidths=0.15, zorder=5, rasterized=True)
        except Exception:
            pass

    format_spatial_ax(ax)
    if col == 0:
        ax.set_ylabel('SnC Density', fontsize=7, fontweight='bold', labelpad=4)

# ── Legend ────────────────────────────────────────────────────────────────────
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#E8E8E8',
           markersize=3, markeredgewidth=0, label='Non-SnC'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#D62728',
           markeredgecolor='black', markeredgewidth=0.3, markersize=4, label='SnC'),
]
fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.99, 0.98),
           frameon=True, fancybox=False, edgecolor='#CCCCCC', fontsize=5.5,
           handletextpad=0.3, borderpad=0.3)

plt.savefig(str(FIGURES_DIR / 'Fig_snc_map_density.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_snc_map_density.svg'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_snc_map_density.pdf / .svg")


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: DBSCAN CLUSTERING (1 row × 4 cols, compact)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print("FIGURE 2: DBSCAN Clustering")
print(f"{'='*60}")

# Compute eps from data
all_nn_dists = []
for sid in samples:
    d = sample_data[sid]
    if len(d['snc_coords']) >= 2:
        tree = KDTree(d['snc_coords'])
        dists, _ = tree.query(d['snc_coords'], k=2)
        all_nn_dists.extend(dists[:, 1])

median_nn = np.median(all_nn_dists)
eps_val = median_nn * 1.5
min_samples_dbscan = 3
print(f"  eps={eps_val:.0f} (1.5 × median NN={median_nn:.0f}), min_samples={min_samples_dbscan}")

cluster_pal = ['#4477AA', '#EE6677', '#228833', '#CCBB44', '#66CCEE',
               '#AA3377', '#BBBBBB', '#EE8866', '#44BB99', '#332288']

fig = plt.figure(figsize=(7.2, 2.2))
gs = gridspec.GridSpec(1, n_samples, wspace=0.06, left=0.02, right=0.98, top=0.85, bottom=0.04)

dbscan_summary = []

for col, sid in enumerate(samples):
    d = sample_data[sid]
    ax = fig.add_subplot(gs[0, col])

    ax.scatter(d['coords'][:, 0], d['coords'][:, 1], c='#EEEEEE', s=1,
               alpha=0.3, edgecolors='none', rasterized=True)

    if len(d['snc_coords']) >= min_samples_dbscan:
        db = DBSCAN(eps=eps_val, min_samples=min_samples_dbscan).fit(d['snc_coords'])
        labels = db.labels_
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()

        # Noise
        noise_mask = labels == -1
        if noise_mask.sum() > 0:
            ax.scatter(d['snc_coords'][noise_mask, 0], d['snc_coords'][noise_mask, 1],
                       c='#AAAAAA', s=6, alpha=0.5, marker='x', linewidths=0.4, rasterized=True)

        # Clusters
        for cl in sorted(set(labels) - {-1}):
            cl_mask = labels == cl
            ax.scatter(d['snc_coords'][cl_mask, 0], d['snc_coords'][cl_mask, 1],
                       c=cluster_pal[cl % len(cluster_pal)], s=10, alpha=0.9,
                       edgecolors='black', linewidths=0.2, rasterized=True)

            cl_coords = d['snc_coords'][cl_mask]
            spread = np.sqrt(np.var(cl_coords[:, 0]) + np.var(cl_coords[:, 1]))
            dbscan_summary.append({
                'sample': d['name'], 'sex': SELECTED[sid]['sex'], 'age': SELECTED[sid]['age'],
                'cluster_id': cl, 'n_spots': cl_mask.sum(), 'spatial_spread': spread,
            })

        subtitle = f"{n_clusters} clusters, {n_noise} noise"
    else:
        subtitle = f"n={len(d['snc_coords'])} SnC"

    format_spatial_ax(ax)
    ax.set_title(f"{d['name']}\n{subtitle}", fontsize=7, fontweight='bold', pad=3)

# Compact legend
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=cluster_pal[0],
           markeredgecolor='black', markeredgewidth=0.3, markersize=3.5, label='Cluster'),
    Line2D([0], [0], marker='x', color='#AAAAAA', markersize=3.5,
           markeredgewidth=0.5, linestyle='None', label='Noise'),
]
fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.99, 0.99),
           frameon=True, fancybox=False, edgecolor='#CCCCCC', fontsize=5.5, ncol=1,
           handletextpad=0.3, borderpad=0.3)

fig.suptitle(f'DBSCAN Clustering (ε={eps_val:.0f}, min_pts={min_samples_dbscan})',
             fontsize=8, fontweight='bold', y=0.97)

plt.savefig(str(FIGURES_DIR / 'Fig_snc_dbscan.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_snc_dbscan.svg'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_snc_dbscan.pdf / .svg")

if dbscan_summary:
    pd.DataFrame(dbscan_summary).to_csv(RESULTS_DIR / 'snc_dbscan_clusters.csv', index=False)
    print(f"✓ Saved: {RESULTS_DIR / 'snc_dbscan_clusters.csv'}")


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 3: NN DISTANCES + RIPLEY'S L (2 panels side by side)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print("FIGURE 3: NN Distances + Ripley's L")
print(f"{'='*60}")

# ── Ripley's L function ──────────────────────────────────────────────────────
def ripleys_L(points, coords_all, radii, n_sim=99):
    n = len(points)
    x_min, x_max = coords_all[:, 0].min(), coords_all[:, 0].max()
    y_min, y_max = coords_all[:, 1].min(), coords_all[:, 1].max()
    area = (x_max - x_min) * (y_max - y_min)
    density = n / area

    if n < 2:
        return radii, np.zeros_like(radii), None, None

    tree = KDTree(points)
    K_obs = np.zeros(len(radii))
    for ri, r in enumerate(radii):
        counts = tree.query_ball_point(points, r)
        total = sum(len(c) - 1 for c in counts)
        K_obs[ri] = total / (n * density)
    L_obs = np.sqrt(K_obs / np.pi) - radii

    L_sims = np.zeros((n_sim, len(radii)))
    for s in range(n_sim):
        rand_pts = np.column_stack([
            np.random.uniform(x_min, x_max, n),
            np.random.uniform(y_min, y_max, n)
        ])
        tree_sim = KDTree(rand_pts)
        K_sim = np.zeros(len(radii))
        for ri, r in enumerate(radii):
            counts = tree_sim.query_ball_point(rand_pts, r)
            total = sum(len(c) - 1 for c in counts)
            K_sim[ri] = total / (n * density)
        L_sims[s] = np.sqrt(K_sim / np.pi) - radii

    return radii, L_obs, np.percentile(L_sims, 2.5, axis=0), np.percentile(L_sims, 97.5, axis=0)

# ── Figure: 3 panels ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(7.2, 2.4))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], wspace=0.35,
                       left=0.07, right=0.97, top=0.85, bottom=0.18)

nn_summary = []

# Panel A: SnC→SnC NN distance boxplot
ax_a = fig.add_subplot(gs[0, 0])
nn_data_box = []
nn_labels_box = []
nn_colors_box = []

for sid in samples:
    d = sample_data[sid]
    if len(d['snc_coords']) >= 2:
        tree = KDTree(d['snc_coords'])
        dists, _ = tree.query(d['snc_coords'], k=2)
        nn_dists = dists[:, 1]
        nn_data_box.append(nn_dists)
        nn_labels_box.append(d['name'])
        nn_colors_box.append(SAMPLE_COLORS[d['name']])

        nn_summary.append({
            'sample': d['name'], 'sex': SELECTED[sid]['sex'], 'age': SELECTED[sid]['age'],
            'type': 'SnC-to-SnC', 'n_snc': len(d['snc_coords']),
            'median_nn': np.median(nn_dists), 'mean_nn': np.mean(nn_dists),
            'std_nn': np.std(nn_dists), 'q25_nn': np.percentile(nn_dists, 25),
            'q75_nn': np.percentile(nn_dists, 75),
        })

bp = ax_a.boxplot(nn_data_box, labels=nn_labels_box, widths=0.55, patch_artist=True,
                  showfliers=False, medianprops=dict(color='black', linewidth=0.8),
                  whiskerprops=dict(linewidth=0.5), capprops=dict(linewidth=0.5),
                  boxprops=dict(linewidth=0.5))
for patch, color in zip(bp['boxes'], nn_colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax_a.set_ylabel('Distance (px)')
ax_a.set_title('SnC→SnC NN Distance', fontweight='bold', fontsize=7.5)
ax_a.tick_params(axis='x', rotation=30)
ax_a.text(-0.15, 1.05, 'A', transform=ax_a.transAxes, fontsize=10, fontweight='bold', va='top')

# Panel B: SnC→SnC NN histogram overlay
ax_b = fig.add_subplot(gs[0, 1])

for sid in samples:
    d = sample_data[sid]
    if len(d['snc_coords']) >= 2:
        tree = KDTree(d['snc_coords'])
        dists, _ = tree.query(d['snc_coords'], k=2)
        nn_dists = dists[:, 1]
        ax_b.hist(nn_dists, bins=25, alpha=0.4, density=True,
                  color=SAMPLE_COLORS[d['name']], label=d['name'], linewidth=0)
        # KDE line
        try:
            kde = gaussian_kde(nn_dists, bw_method=0.3)
            x_kde = np.linspace(0, nn_dists.max() * 1.1, 200)
            ax_b.plot(x_kde, kde(x_kde), color=SAMPLE_COLORS[d['name']], linewidth=1.2)
        except Exception:
            pass

ax_b.set_xlabel('Distance (px)')
ax_b.set_ylabel('Density')
ax_b.set_title('NN Distance Distribution', fontweight='bold', fontsize=7.5)
ax_b.legend(fontsize=5, frameon=True, fancybox=False, edgecolor='#CCCCCC',
            handletextpad=0.3, borderpad=0.3, loc='upper right')
ax_b.text(-0.15, 1.05, 'B', transform=ax_b.transAxes, fontsize=10, fontweight='bold', va='top')

# Panel C: Ripley's L
ax_c = fig.add_subplot(gs[0, 2])

for sid in samples:
    d = sample_data[sid]
    if len(d['snc_coords']) >= 10:
        max_r = min(np.ptp(d['coords'][:, 0]), np.ptp(d['coords'][:, 1])) / 4
        radii = np.linspace(10, max_r, 30)
        r, L_obs, L_lo, L_hi = ripleys_L(d['snc_coords'], d['coords'], radii, n_sim=99)

        ax_c.plot(r, L_obs, color=SAMPLE_COLORS[d['name']], linewidth=1.2, label=d['name'])

        sig = np.any(L_obs > L_hi)
        print(f"  {d['name']}: {'CLUSTERED' if sig else 'random'} "
              f"(max L={np.max(L_obs):.0f}, envelope={np.max(L_hi):.0f})")

# Plot one CSR envelope (from last sample, representative)
if L_hi is not None:
    ax_c.fill_between(r, L_lo, L_hi, alpha=0.15, color='gray', label='95% CSR')

ax_c.axhline(0, color='black', linewidth=0.4, linestyle=':')
ax_c.set_xlabel('Radius (r)')
ax_c.set_ylabel('L(r) − r')
ax_c.set_title("Ripley's L Function", fontweight='bold', fontsize=7.5)
ax_c.legend(fontsize=5, frameon=True, fancybox=False, edgecolor='#CCCCCC',
            handletextpad=0.3, borderpad=0.3, loc='upper left')
ax_c.text(-0.15, 1.05, 'C', transform=ax_c.transAxes, fontsize=10, fontweight='bold', va='top')

plt.savefig(str(FIGURES_DIR / 'Fig_snc_nn_ripley.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_snc_nn_ripley.svg'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_snc_nn_ripley.pdf / .svg")


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 4: COMPACT SUMMARY — SnC map + DBSCAN + Dominant CT (3 rows × 4 cols)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print("FIGURE 4: Combined Summary Panel")
print(f"{'='*60}")

ct_colors = {
    'Excitatory':      '#2ca02c', 'Inhibitory':      '#ff7f0e',
    'Oligodendrocyte': '#9467bd', 'Astrocyte':       '#1f77b4',
    'Microglia':       '#8c564b', 'OPC':             '#e377c2',
    'Endothelial':     '#d62728', 'Pericyte':        '#17becf',
    'PVM':             '#bcbd22', 'VLMC':            '#f7b6d2',
    'VSMC':            '#7f7f7f', 'Adaptive':        '#aec7e8',
    'Vascular':        '#17becf',
}

present_cts = sorted(adata_sel.obs['dominant_celltype'].unique())

fig = plt.figure(figsize=(7.2, 5.6))
gs = gridspec.GridSpec(3, n_samples + 1, width_ratios=[1]*n_samples + [0.12],
                       hspace=0.14, wspace=0.06,
                       left=0.06, right=0.93, top=0.93, bottom=0.03)

row_labels = ['SnC Map', 'DBSCAN', 'Cell Type']

for col, sid in enumerate(samples):
    d = sample_data[sid]
    mask = (adata_sel.obs['sample_id'] == sid).values

    # ── Row 0: SnC map ──────────────────────────────────────────────────────
    ax = fig.add_subplot(gs[0, col])
    ax.scatter(d['coords'][:, 0], d['coords'][:, 1], c='#E8E8E8', s=1.5,
               alpha=0.4, edgecolors='none', rasterized=True)
    ax.scatter(d['snc_coords'][:, 0], d['snc_coords'][:, 1], c='#D62728',
               s=8, alpha=0.9, edgecolors='black', linewidths=0.2, rasterized=True, zorder=3)
    format_spatial_ax(ax)
    n_snc = d['is_snc'].sum()
    ax.set_title(f"{d['name']} (age {SELECTED[sid]['age']})", fontsize=7, fontweight='bold', pad=3)
    ax.text(0.5, -0.02, f'{n_snc} SnC ({n_snc/len(d["is_snc"])*100:.1f}%)',
            transform=ax.transAxes, ha='center', fontsize=5, color='#555555', style='italic')
    if col == 0:
        ax.set_ylabel(row_labels[0], fontsize=7, fontweight='bold', labelpad=4)

    # ── Row 1: DBSCAN ───────────────────────────────────────────────────────
    ax = fig.add_subplot(gs[1, col])
    ax.scatter(d['coords'][:, 0], d['coords'][:, 1], c='#EEEEEE', s=1,
               alpha=0.3, edgecolors='none', rasterized=True)

    if len(d['snc_coords']) >= min_samples_dbscan:
        db = DBSCAN(eps=eps_val, min_samples=min_samples_dbscan).fit(d['snc_coords'])
        labels = db.labels_
        n_cl = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()

        noise_mask = labels == -1
        if noise_mask.sum() > 0:
            ax.scatter(d['snc_coords'][noise_mask, 0], d['snc_coords'][noise_mask, 1],
                       c='#AAAAAA', s=5, alpha=0.5, marker='x', linewidths=0.3, rasterized=True)
        for cl in sorted(set(labels) - {-1}):
            cl_mask = labels == cl
            ax.scatter(d['snc_coords'][cl_mask, 0], d['snc_coords'][cl_mask, 1],
                       c=cluster_pal[cl % len(cluster_pal)], s=8, alpha=0.9,
                       edgecolors='black', linewidths=0.2, rasterized=True)
        ax.text(0.5, -0.02, f'{n_cl} clusters, {n_noise} noise',
                transform=ax.transAxes, ha='center', fontsize=5, color='#555555', style='italic')

    format_spatial_ax(ax)
    if col == 0:
        ax.set_ylabel(row_labels[1], fontsize=7, fontweight='bold', labelpad=4)

    # ── Row 2: Dominant cell type ────────────────────────────────────────────
    ax = fig.add_subplot(gs[2, col])
    dom = adata_sel.obs.loc[mask, 'dominant_celltype'].values
    for ct in present_cts:
        ct_mask = dom == ct
        if ct_mask.sum() > 0:
            ax.scatter(d['coords'][ct_mask, 0], d['coords'][ct_mask, 1],
                       c=ct_colors.get(ct, '#999'), s=1.5, alpha=0.85,
                       edgecolors='none', rasterized=True)
    format_spatial_ax(ax)
    if col == 0:
        ax.set_ylabel(row_labels[2], fontsize=7, fontweight='bold', labelpad=4)

# ── Legends ──────────────────────────────────────────────────────────────────
# Row 0 legend: SnC
leg_ax0 = fig.add_subplot(gs[0, n_samples])
leg_ax0.axis('off')
handles_snc = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#E8E8E8',
           markersize=3, markeredgewidth=0, label='Non-SnC'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#D62728',
           markeredgecolor='black', markeredgewidth=0.3, markersize=3.5, label='SnC'),
]
leg_ax0.legend(handles=handles_snc, loc='center left', frameon=False,
               fontsize=5, labelspacing=0.6, handletextpad=0.2, borderpad=0)

# Row 1 legend: DBSCAN
leg_ax1 = fig.add_subplot(gs[1, n_samples])
leg_ax1.axis('off')
handles_db = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=cluster_pal[0],
           markeredgecolor='black', markeredgewidth=0.3, markersize=3, label='Cluster'),
    Line2D([0], [0], marker='x', color='#AAAAAA', markersize=3,
           markeredgewidth=0.5, linestyle='None', label='Noise'),
]
leg_ax1.legend(handles=handles_db, loc='center left', frameon=False,
               fontsize=5, labelspacing=0.6, handletextpad=0.2, borderpad=0)

# Row 2 legend: Cell types
leg_ax2 = fig.add_subplot(gs[2, n_samples])
leg_ax2.axis('off')
handles_ct = [Line2D([0], [0], marker='o', color='w',
                     markerfacecolor=ct_colors.get(ct, '#999'),
                     markersize=3, markeredgewidth=0, label=ct)
              for ct in present_cts]
leg_ax2.legend(handles=handles_ct, loc='center left', frameon=False,
               fontsize=4.5, labelspacing=0.5, handletextpad=0.2, borderpad=0)

plt.savefig(str(FIGURES_DIR / 'Fig_snc_summary_panel.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_snc_summary_panel.svg'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_snc_summary_panel.pdf / .svg")


# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print("SPATIAL CLUSTERING SUMMARY")
print(f"{'='*60}")

print(f"\n  {'Sample':<8s} {'Sex':>3s} {'Age':>4s} {'SnC':>5s} {'%SnC':>6s} "
      f"{'Clust':>5s} {'med_NN':>7s} {'Ripley':>10s}")
print(f"  {'─'*52}")

for sid in samples:
    d = sample_data[sid]
    meta = SELECTED[sid]
    n_snc = d['is_snc'].sum()
    pct = n_snc / len(d['is_snc']) * 100

    if len(d['snc_coords']) >= min_samples_dbscan:
        db = DBSCAN(eps=eps_val, min_samples=min_samples_dbscan).fit(d['snc_coords'])
        n_cl = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    else:
        n_cl = 0

    nn_row = [r for r in nn_summary if r['sample'] == d['name'] and r['type'] == 'SnC-to-SnC']
    med_nn = f"{nn_row[0]['median_nn']:.0f}" if nn_row else 'N/A'

    if len(d['snc_coords']) >= 10:
        max_r = min(np.ptp(d['coords'][:, 0]), np.ptp(d['coords'][:, 1])) / 4
        radii = np.linspace(10, max_r, 30)
        _, L_obs, _, L_hi = ripleys_L(d['snc_coords'], d['coords'], radii, n_sim=99)
        ripley = "CLUSTERED" if np.any(L_obs > L_hi) else "random"
    else:
        ripley = 'N/A'

    print(f"  {d['name']:<8s} {meta['sex']:>3s} {meta['age']:>4d} {n_snc:>5d} {pct:>5.1f}% "
          f"{n_cl:>5d} {med_nn:>7s} {ripley:>10s}")

# ── Save ─────────────────────────────────────────────────────────────────────
if nn_summary:
    pd.DataFrame(nn_summary).to_csv(RESULTS_DIR / 'snc_nn_distances.csv', index=False)
    print(f"\n✓ Saved: {RESULTS_DIR / 'snc_nn_distances.csv'}")

print(f"\n✓ All figures saved to: {FIGURES_DIR}")
print(f"✓ All results saved to: {RESULTS_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 11: SnC CLUSTER CELL TYPE COMPOSITION
# ═══════════════════════════════════════════════════════════════════════════════

from scipy.stats import fisher_exact

def format_spatial_ax(ax):
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)
        spine.set_edgecolor('#444444')

# ── DBSCAN params (same as Cell 10) ─────────────────────────────────────────
all_nn_dists = []
for sid in samples:
    mask = (adata_sel.obs['sample_id'] == sid).values
    is_snc = (adata_sel.obs.loc[mask, 'is_senescent'] == 1).values
    snc_coords = adata_sel[mask].obsm['spatial'][is_snc]
    if len(snc_coords) >= 2:
        tree = KDTree(snc_coords)
        dists, _ = tree.query(snc_coords, k=2)
        all_nn_dists.extend(dists[:, 1])

median_nn = np.median(all_nn_dists)
eps_val = median_nn * 1.5
min_samples_dbscan = 3
print(f"  DBSCAN: eps={eps_val:.0f}, min_samples={min_samples_dbscan}")


# ═══════════════════════════════════════════════════════════════════════════════
# 1. CELL TYPE COMPOSITION: SnC CLUSTERED vs SnC NOISE vs NON-SnC
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print("1. CELL TYPE COMPOSITION BY SnC STATUS")
print(f"{'='*60}")

present_cts = sorted(adata_sel.obs['dominant_celltype'].unique())
composition_rows = []

for sid in samples:
    name, _ = get_meta(sid)
    meta = SELECTED[sid]
    mask = (adata_sel.obs['sample_id'] == sid).values
    coords = adata_sel[mask].obsm['spatial']
    is_snc = (adata_sel.obs.loc[mask, 'is_senescent'] == 1).values
    dom = adata_sel.obs.loc[mask, 'dominant_celltype'].values
    snc_coords = coords[is_snc]

    # DBSCAN labels
    if len(snc_coords) >= min_samples_dbscan:
        db = DBSCAN(eps=eps_val, min_samples=min_samples_dbscan).fit(snc_coords)
        labels = db.labels_
    else:
        labels = np.full(len(snc_coords), -1)

    # Assign categories
    spot_category = np.full(mask.sum(), 'Non-SnC', dtype=object)
    snc_idx = np.where(is_snc)[0]
    for j, idx in enumerate(snc_idx):
        if labels[j] == -1:
            spot_category[idx] = 'SnC-Noise'
        else:
            spot_category[idx] = f'SnC-Cluster'

    print(f"\n  {name} (age {meta['age']}, {meta['sex']})")
    print(f"  {'Category':<16s}", end='')
    for ct in present_cts:
        print(f" {ct[:10]:>10s}", end='')
    print(f" {'Total':>7s}")
    print(f"  {'─'*(16 + 10*len(present_cts) + len(present_cts) + 7)}")

    for cat in ['SnC-Cluster', 'SnC-Noise', 'Non-SnC']:
        cat_mask = spot_category == cat
        n_cat = cat_mask.sum()
        if n_cat == 0:
            continue
        cat_dom = dom[cat_mask]
        print(f"  {cat:<16s}", end='')
        for ct in present_cts:
            n = (cat_dom == ct).sum()
            pct = n / n_cat * 100
            print(f" {pct:>9.1f}%", end='')
            composition_rows.append({
                'sample': name, 'sex': meta['sex'], 'age': meta['age'],
                'category': cat, 'cell_type': ct, 'n_spots': n,
                'pct': pct, 'total_in_category': n_cat,
            })
        print(f" {n_cat:>7d}")


# ═══════════════════════════════════════════════════════════════════════════════
# 2. ENRICHMENT / DEPLETION: SnC vs BACKGROUND
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print("2. CELL TYPE ENRICHMENT IN SnC vs BACKGROUND")
print(f"{'='*60}")
print("  Log2(fold change) = log2( %ct in SnC / %ct in Non-SnC )")
print("  >0 = enriched in SnC, <0 = depleted")

enrichment_rows = []

for sid in samples:
    name, _ = get_meta(sid)
    meta = SELECTED[sid]
    mask = (adata_sel.obs['sample_id'] == sid).values
    is_snc = (adata_sel.obs.loc[mask, 'is_senescent'] == 1).values
    dom = adata_sel.obs.loc[mask, 'dominant_celltype'].values

    snc_dom = dom[is_snc]
    non_dom = dom[~is_snc]
    n_snc = len(snc_dom)
    n_non = len(non_dom)

    print(f"\n  {name} (age {meta['age']})")
    print(f"  {'Cell Type':<16s} {'%SnC':>7s} {'%Bkgd':>7s} {'log2FC':>8s} {'p-value':>10s} {'Sig':>5s}")
    print(f"  {'─'*56}")

    for ct in present_cts:
        n_ct_snc = (snc_dom == ct).sum()
        n_ct_non = (non_dom == ct).sum()
        pct_snc = n_ct_snc / n_snc * 100 if n_snc > 0 else 0
        pct_non = n_ct_non / n_non * 100 if n_non > 0 else 0

        # Log2 fold change with pseudocount
        fc = (pct_snc + 0.1) / (pct_non + 0.1)
        log2fc = np.log2(fc)

        # Fisher's exact test
        table = np.array([
            [n_ct_snc, n_snc - n_ct_snc],
            [n_ct_non, n_non - n_ct_non]
        ])
        if table.min() >= 0 and table.sum() > 0:
            _, pval = fisher_exact(table)
        else:
            pval = 1.0

        sig = '*' if pval < 0.05 else ''
        if pval < 0.01:
            sig = '**'
        if pval < 0.001:
            sig = '***'

        print(f"  {ct:<16s} {pct_snc:>6.1f}% {pct_non:>6.1f}% {log2fc:>+8.2f} {pval:>10.3e} {sig:>5s}")

        enrichment_rows.append({
            'sample': name, 'sex': meta['sex'], 'age': meta['age'],
            'cell_type': ct, 'pct_snc': pct_snc, 'pct_background': pct_non,
            'log2fc': log2fc, 'pval': pval,
        })


# ═══════════════════════════════════════════════════════════════════════════════
# 3. PER-CLUSTER CELL TYPE COMPOSITION
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print("3. PER-CLUSTER CELL TYPE COMPOSITION")
print(f"{'='*60}")
print("  Are clusters homogeneous (one cell type) or mixed?")

cluster_detail_rows = []

for sid in samples:
    name, _ = get_meta(sid)
    meta = SELECTED[sid]
    mask = (adata_sel.obs['sample_id'] == sid).values
    coords = adata_sel[mask].obsm['spatial']
    is_snc = (adata_sel.obs.loc[mask, 'is_senescent'] == 1).values
    dom = adata_sel.obs.loc[mask, 'dominant_celltype'].values
    snc_coords = coords[is_snc]
    snc_dom = dom[is_snc]

    if len(snc_coords) < min_samples_dbscan:
        continue

    db = DBSCAN(eps=eps_val, min_samples=min_samples_dbscan).fit(snc_coords)
    labels = db.labels_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    print(f"\n  {name} (age {meta['age']}) — {n_clusters} clusters")

    for cl in sorted(set(labels) - {-1}):
        cl_mask = labels == cl
        cl_dom = snc_dom[cl_mask]
        n_cl = cl_mask.sum()

        cts_in_cluster = pd.Series(cl_dom).value_counts()
        dominant_ct = cts_in_cluster.index[0]
        dominant_pct = cts_in_cluster.iloc[0] / n_cl * 100
        n_types = len(cts_in_cluster)

        # Shannon entropy for heterogeneity
        props = cts_in_cluster.values / n_cl
        entropy = -np.sum(props * np.log2(props + 1e-10))
        max_entropy = np.log2(n_types) if n_types > 1 else 1
        evenness = entropy / max_entropy if max_entropy > 0 else 0

        homogeneity = "homogeneous" if dominant_pct >= 80 else ("mixed" if dominant_pct >= 50 else "heterogeneous")

        ct_str = ', '.join([f"{ct}({int(n)})" for ct, n in cts_in_cluster.items()])
        print(f"    Cluster {cl}: n={n_cl}, dominant={dominant_ct} ({dominant_pct:.0f}%), "
              f"{homogeneity}, entropy={entropy:.2f}")
        print(f"      Composition: {ct_str}")

        cluster_detail_rows.append({
            'sample': name, 'sex': meta['sex'], 'age': meta['age'],
            'cluster_id': cl, 'n_spots': n_cl,
            'dominant_ct': dominant_ct, 'dominant_pct': dominant_pct,
            'n_cell_types': n_types, 'entropy': entropy, 'evenness': evenness,
            'homogeneity': homogeneity, 'composition': ct_str,
        })


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: STACKED BAR — CELL TYPE COMPOSITION BY SnC STATUS
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print("PLOTTING")
print(f"{'='*60}")

fig = plt.figure(figsize=(7.2, 2.8))
gs = gridspec.GridSpec(1, n_samples + 1, width_ratios=[1]*n_samples + [0.35],
                       wspace=0.25, left=0.06, right=0.88, top=0.85, bottom=0.18)

categories = ['SnC-Cluster', 'SnC-Noise', 'Non-SnC']
cat_labels = ['SnC\nCluster', 'SnC\nNoise', 'Non-\nSnC']

for col, sid in enumerate(samples):
    name, _ = get_meta(sid)
    meta = SELECTED[sid]
    mask = (adata_sel.obs['sample_id'] == sid).values
    coords = adata_sel[mask].obsm['spatial']
    is_snc = (adata_sel.obs.loc[mask, 'is_senescent'] == 1).values
    dom = adata_sel.obs.loc[mask, 'dominant_celltype'].values
    snc_coords = coords[is_snc]

    if len(snc_coords) >= min_samples_dbscan:
        db = DBSCAN(eps=eps_val, min_samples=min_samples_dbscan).fit(snc_coords)
        labels = db.labels_
    else:
        labels = np.full(len(snc_coords), -1)

    spot_category = np.full(mask.sum(), 'Non-SnC', dtype=object)
    snc_idx = np.where(is_snc)[0]
    for j, idx in enumerate(snc_idx):
        spot_category[idx] = 'SnC-Noise' if labels[j] == -1 else 'SnC-Cluster'

    ax = fig.add_subplot(gs[0, col])
    x = np.arange(len(categories))
    bottom = np.zeros(len(categories))

    for ct in present_cts:
        fracs = []
        for cat in categories:
            cat_mask = spot_category == cat
            n_cat = cat_mask.sum()
            if n_cat > 0:
                fracs.append((dom[cat_mask] == ct).sum() / n_cat * 100)
            else:
                fracs.append(0)
        fracs = np.array(fracs)
        ax.bar(x, fracs, bottom=bottom, width=0.65, color=ct_colors.get(ct, '#999'),
               edgecolor='white', linewidth=0.3, label=ct if col == 0 else None)
        bottom += fracs

    ax.set_xticks(x)
    ax.set_xticklabels(cat_labels, fontsize=5.5)
    ax.set_ylim(0, 100)
    if col == 0:
        ax.set_ylabel('% of spots')
    else:
        ax.set_yticklabels([])
    ax.set_title(f"{name}\n(age {meta['age']})", fontsize=7, fontweight='bold')

    # Annotate n
    for i, cat in enumerate(categories):
        n = (spot_category == cat).sum()
        ax.text(i, 101, f'n={n}', ha='center', fontsize=4.5, color='#666666')

# Legend
leg_ax = fig.add_subplot(gs[0, n_samples])
leg_ax.axis('off')
handles = [plt.Rectangle((0,0), 1, 1, facecolor=ct_colors.get(ct, '#999'),
                          edgecolor='white', linewidth=0.3) for ct in present_cts]
leg_ax.legend(handles, present_cts, loc='center left', frameon=False,
              fontsize=5, labelspacing=0.5, handletextpad=0.3, borderpad=0)

fig.suptitle('Cell Type Composition: SnC Clusters vs Noise vs Background',
             fontsize=8, fontweight='bold')
plt.savefig(str(FIGURES_DIR / 'Fig_snc_cluster_celltype_composition.pdf'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_snc_cluster_celltype_composition.svg'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_snc_cluster_celltype_composition.pdf / .svg")


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: ENRICHMENT HEATMAP — log2FC of cell types in SnC vs background
# ═══════════════════════════════════════════════════════════════════════════════

df_enrich = pd.DataFrame(enrichment_rows)

# Pivot: rows = cell types, cols = samples
pivot_fc = df_enrich.pivot(index='cell_type', columns='sample', values='log2fc')
pivot_pval = df_enrich.pivot(index='cell_type', columns='sample', values='pval')

# Order samples
sample_order = [SELECTED[sid]['name'] for sid in samples]
pivot_fc = pivot_fc[sample_order]
pivot_pval = pivot_pval[sample_order]

fig, ax = plt.subplots(figsize=(3.8, 3.2))

# Symmetric color range
vmax = max(abs(pivot_fc.values.min()), abs(pivot_fc.values.max()), 1.5)
im = ax.imshow(pivot_fc.values, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)

# Significance annotations
for i in range(pivot_fc.shape[0]):
    for j in range(pivot_fc.shape[1]):
        fc = pivot_fc.values[i, j]
        pv = pivot_pval.values[i, j]
        sig = ''
        if pv < 0.001:
            sig = '***'
        elif pv < 0.01:
            sig = '**'
        elif pv < 0.05:
            sig = '*'

        # Value text
        txt_color = 'white' if abs(fc) > vmax * 0.6 else 'black'
        ax.text(j, i, f'{fc:+.1f}', ha='center', va='center', fontsize=5.5, color=txt_color)
        if sig:
            ax.text(j, i + 0.3, sig, ha='center', va='center', fontsize=5, color=txt_color,
                    fontweight='bold')

ax.set_xticks(range(len(sample_order)))
ax.set_xticklabels(sample_order, fontsize=6.5)
ax.set_yticks(range(len(pivot_fc.index)))
ax.set_yticklabels(pivot_fc.index, fontsize=6.5)
ax.set_title('SnC Enrichment (log₂FC vs background)', fontsize=8, fontweight='bold', pad=8)

cbar = plt.colorbar(im, ax=ax, shrink=0.7, aspect=15)
cbar.ax.tick_params(labelsize=5.5)
cbar.set_label('log₂(fold change)', fontsize=6)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'Fig_snc_enrichment_heatmap.pdf'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_snc_enrichment_heatmap.svg'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_snc_enrichment_heatmap.pdf / .svg")


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 3: SPATIAL MAP — SnC COLORED BY DOMINANT CELL TYPE
# ═══════════════════════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(7.2, 2.2))
gs = gridspec.GridSpec(1, n_samples + 1, width_ratios=[1]*n_samples + [0.15],
                       wspace=0.06, left=0.02, right=0.93, top=0.85, bottom=0.04)

for col, sid in enumerate(samples):
    name, _ = get_meta(sid)
    meta = SELECTED[sid]
    mask = (adata_sel.obs['sample_id'] == sid).values
    coords = adata_sel[mask].obsm['spatial']
    is_snc = (adata_sel.obs.loc[mask, 'is_senescent'] == 1).values
    dom = adata_sel.obs.loc[mask, 'dominant_celltype'].values

    snc_coords = coords[is_snc]
    snc_dom = dom[is_snc]

    ax = fig.add_subplot(gs[0, col])
    # Background tissue
    ax.scatter(coords[:, 0], coords[:, 1], c='#EEEEEE', s=1, alpha=0.3,
               edgecolors='none', rasterized=True)
    # SnC colored by cell type
    for ct in present_cts:
        ct_mask = snc_dom == ct
        if ct_mask.sum() > 0:
            ax.scatter(snc_coords[ct_mask, 0], snc_coords[ct_mask, 1],
                       c=ct_colors.get(ct, '#999'), s=12, alpha=0.9,
                       edgecolors='black', linewidths=0.2, rasterized=True, zorder=3)

    format_spatial_ax(ax)
    n_snc = is_snc.sum()
    ax.set_title(f"{name} (age {meta['age']})\n{n_snc} SnC", fontsize=7, fontweight='bold', pad=3)

# Legend
leg_ax = fig.add_subplot(gs[0, n_samples])
leg_ax.axis('off')
# Only show cell types present in SnC
snc_cts_all = adata_sel.obs.loc[adata_sel.obs['is_senescent'] == 1, 'dominant_celltype'].unique()
handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=ct_colors.get(ct, '#999'),
                  markeredgecolor='black', markeredgewidth=0.2, markersize=4, label=ct)
           for ct in present_cts if ct in snc_cts_all]
leg_ax.legend(handles=handles, loc='center left', frameon=False,
              fontsize=5, labelspacing=0.6, handletextpad=0.2, borderpad=0)

fig.suptitle('SnC Spots Colored by Dominant Cell Type', fontsize=8, fontweight='bold')
plt.savefig(str(FIGURES_DIR / 'Fig_snc_celltype_spatial.pdf'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_snc_celltype_spatial.svg'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_snc_celltype_spatial.pdf / .svg")


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 4: PER-CLUSTER COMPOSITION — MINI PIES OR STACKED BARS
# ═══════════════════════════════════════════════════════════════════════════════

if cluster_detail_rows:
    df_clusters = pd.DataFrame(cluster_detail_rows)

    fig, axes = plt.subplots(1, n_samples, figsize=(7.2, 2.5))
    if n_samples == 1:
        axes = [axes]

    for col, sid in enumerate(samples):
        name, _ = get_meta(sid)
        meta = SELECTED[sid]
        mask = (adata_sel.obs['sample_id'] == sid).values
        coords = adata_sel[mask].obsm['spatial']
        is_snc = (adata_sel.obs.loc[mask, 'is_senescent'] == 1).values
        dom = adata_sel.obs.loc[mask, 'dominant_celltype'].values
        snc_coords = coords[is_snc]
        snc_dom = dom[is_snc]

        ax = axes[col]
        ax.scatter(coords[:, 0], coords[:, 1], c='#F0F0F0', s=1, alpha=0.3,
                   edgecolors='none', rasterized=True)

        if len(snc_coords) >= min_samples_dbscan:
            db = DBSCAN(eps=eps_val, min_samples=min_samples_dbscan).fit(snc_coords)
            labels = db.labels_

            # Noise as small gray x
            noise = labels == -1
            if noise.sum() > 0:
                ax.scatter(snc_coords[noise, 0], snc_coords[noise, 1],
                           c='#BBBBBB', s=4, marker='x', linewidths=0.3, alpha=0.5, rasterized=True)

            # Clustered SnC colored by cell type
            for cl in sorted(set(labels) - {-1}):
                cl_mask = labels == cl
                cl_coords = snc_coords[cl_mask]
                cl_dom = snc_dom[cl_mask]

                for ct in present_cts:
                    ct_in_cl = cl_dom == ct
                    if ct_in_cl.sum() > 0:
                        ax.scatter(cl_coords[ct_in_cl, 0], cl_coords[ct_in_cl, 1],
                                   c=ct_colors.get(ct, '#999'), s=14, alpha=0.9,
                                   edgecolors='black', linewidths=0.3, rasterized=True, zorder=3)

                # Cluster label at centroid
                cx, cy = cl_coords[:, 0].mean(), cl_coords[:, 1].mean()
                ax.text(cx, cy - 30, f'C{cl}', ha='center', fontsize=4.5,
                        fontweight='bold', color='#333333', zorder=6,
                        bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                                  edgecolor='#999999', linewidth=0.3, alpha=0.8))

        format_spatial_ax(ax)
        ax.set_title(f"{name} (age {meta['age']})", fontsize=7, fontweight='bold', pad=3)

    fig.suptitle('SnC Clusters: Cell Type Identity', fontsize=8, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / 'Fig_snc_cluster_celltype_spatial.pdf'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(str(FIGURES_DIR / 'Fig_snc_cluster_celltype_spatial.svg'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print("✓ Fig_snc_cluster_celltype_spatial.pdf / .svg")


# ═══════════════════════════════════════════════════════════════════════════════
# SAVE ALL RESULTS
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print("SAVING RESULTS")
print(f"{'='*60}")

pd.DataFrame(composition_rows).to_csv(
    RESULTS_DIR / 'snc_cluster_celltype_composition.csv', index=False)
print(f"  ✓ {RESULTS_DIR / 'snc_cluster_celltype_composition.csv'}")

pd.DataFrame(enrichment_rows).to_csv(
    RESULTS_DIR / 'snc_celltype_enrichment.csv', index=False)
print(f"  ✓ {RESULTS_DIR / 'snc_celltype_enrichment.csv'}")

if cluster_detail_rows:
    pd.DataFrame(cluster_detail_rows).to_csv(
        RESULTS_DIR / 'snc_cluster_detail.csv', index=False)
    print(f"  ✓ {RESULTS_DIR / 'snc_cluster_detail.csv'}")

print(f"\n✓ All figures: {FIGURES_DIR}")
print(f"✓ All results: {RESULTS_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 12: MULTI-PANEL — Module Scores × SnC Cell-Type + DBSCAN Clusters
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Circle as MplCircle
from matplotlib.colors import Normalize
from sklearn.cluster import DBSCAN
from scipy.spatial import KDTree
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
score_cols = ['Score_SASP', 'Score_p53_Targets', 'Score_DDR',
              'Score_LysosomalContent', 'Score_SenMayo', 'Score_Fridman_Up']
score_labels = ['SASP', 'p53 Targets', 'DDR', 'Lysosomal Content', 'SenMayo', 'Fridman Up']

ct_colors = {
    'Excitatory': '#2ca02c', 'Inhibitory': '#ff7f0e',
    'Oligodendrocyte': '#9467bd', 'Astrocyte': '#1f77b4',
    'Microglia': '#8c564b', 'OPC': '#e377c2',
    'Endothelial': '#d62728', 'Pericyte': '#17becf',
    'PVM': '#bcbd22', 'VLMC': '#f7b6d2',
    'VSMC': '#7f7f7f', 'Adaptive': '#aec7e8',
    'Vascular': '#17becf',
}

# Marker shapes per cluster
cluster_markers = ['o', '^', 's', 'D', 'P', '*', 'X', 'v', '<', '>']
cluster_pal = ['#4477AA', '#EE6677', '#228833', '#CCBB44', '#66CCEE',
               '#AA3377', '#BBBBBB', '#EE8866', '#44BB99', '#332288']

# ── DBSCAN params (same as Cell 10) ──────────────────────────────────────────
all_nn_dists = []
for sid in samples:
    d = sample_data[sid]
    if len(d['snc_coords']) >= 2:
        tree = KDTree(d['snc_coords'])
        dists, _ = tree.query(d['snc_coords'], k=2)
        all_nn_dists.extend(dists[:, 1])
median_nn = np.median(all_nn_dists)
eps_val = median_nn * 1.5
min_samples_dbscan = 3

# ── Pre-compute DBSCAN labels per sample ──────────────────────────────────────
dbscan_labels = {}
for sid in samples:
    d = sample_data[sid]
    if len(d['snc_coords']) >= min_samples_dbscan:
        db = DBSCAN(eps=eps_val, min_samples=min_samples_dbscan).fit(d['snc_coords'])
        dbscan_labels[sid] = db.labels_
    else:
        dbscan_labels[sid] = np.full(len(d['snc_coords']), -1)

# ── Pre-compute global vmin/vmax per score (for consistent colorbars) ────────
score_ranges = {}
for sc_col in score_cols:
    all_vals = []
    for sid in samples:
        mask = (adata_sel.obs['sample_id'] == sid).values
        all_vals.extend(adata_sel.obs.loc[mask, sc_col].values)
    score_ranges[sc_col] = (np.percentile(all_vals, 2), np.percentile(all_vals, 98))

# ── Layout: 13 rows × (n_samples + 1 colorbar col) ──────────────────────────
n_rows = 1 + len(score_cols) * 2  # 13
row_h = 1.8
fig_h = n_rows * row_h + 1.0

fig = plt.figure(figsize=(7.5, fig_h))
gs = gridspec.GridSpec(n_rows, n_samples + 1,
                       width_ratios=[1]*n_samples + [0.03],
                       hspace=0.12, wspace=0.08,
                       left=0.02, right=0.95, top=0.97, bottom=0.03)

present_cts = sorted(adata_sel.obs['dominant_celltype'].unique())

for col, sid in enumerate(samples):
    d = sample_data[sid]
    mask = (adata_sel.obs['sample_id'] == sid).values
    obs_s = adata_sel.obs.loc[mask]
    coords = d['coords']
    snc_mask = d['is_snc']
    snc_coords = d['snc_coords']
    labels = dbscan_labels[sid]
    dom_ct = obs_s['dominant_celltype'].values

    # ── Row 0: Cell type map ─────────────────────────────────────────────────
    ax = fig.add_subplot(gs[0, col])
    for ct in present_cts:
        ct_m = dom_ct == ct
        if ct_m.sum() > 0:
            ax.scatter(coords[ct_m, 0], coords[ct_m, 1],
                       c=ct_colors.get(ct, '#999'), s=4, alpha=0.85,
                       edgecolors='none', rasterized=True)
    format_spatial_ax(ax)
    ax.set_title(f"{d['name']}", fontsize=7, fontweight='bold', pad=3)
    if col == 0:
        ax.set_ylabel('Cell Type', fontsize=6, fontweight='bold', labelpad=4)

    # ── Score rows ───────────────────────────────────────────────────────────
    for si, (sc_col, sc_lab) in enumerate(zip(score_cols, score_labels)):
        row_clean = 1 + si * 2
        row_overlay = 1 + si * 2 + 1

        scores = obs_s[sc_col].values
        vmin, vmax = score_ranges[sc_col]

        # ── Clean heatmap ────────────────────────────────────────────────────
        ax = fig.add_subplot(gs[row_clean, col])
        sc_plot = ax.scatter(coords[:, 0], coords[:, 1], c=scores, cmap='RdYlBu_r',
                             s=4, alpha=0.85, vmin=vmin, vmax=vmax,
                             edgecolors='none', rasterized=True)
        format_spatial_ax(ax)
        if col == 0:
            ax.set_ylabel(sc_lab, fontsize=5.5, fontweight='bold', labelpad=4)

        # Colorbar (once per score, last sample col)
        if col == n_samples - 1:
            cax = fig.add_subplot(gs[row_clean, n_samples])
            cb = fig.colorbar(sc_plot, cax=cax)
            cb.ax.tick_params(labelsize=4, length=1.5, width=0.4)
            cb.outline.set_linewidth(0.4)

        # ── Overlay: dimmed score + SnC (cell-type colored, shape per cluster)
        ax = fig.add_subplot(gs[row_overlay, col])
        ax.scatter(coords[:, 0], coords[:, 1], c=scores, cmap='RdYlBu_r',
                   s=4, alpha=0.25, vmin=vmin, vmax=vmax,
                   edgecolors='none', rasterized=True)

        snc_ct = dom_ct[snc_mask]
        unique_labels = sorted(set(labels) - {-1})

        # Noise spots (cluster = -1): small x
        noise_m = labels == -1
        if noise_m.sum() > 0:
            for ct in np.unique(snc_ct[noise_m]):
                ct_noise = noise_m & (snc_ct == ct)
                if ct_noise.sum() > 0:
                    ax.scatter(snc_coords[ct_noise, 0], snc_coords[ct_noise, 1],
                               c=ct_colors.get(ct, '#999'), s=14, alpha=0.9,
                               marker='x', linewidths=0.5, zorder=5, rasterized=True)

        # Cluster spots: colored by cell type, shaped by cluster ID
        for cl in unique_labels:
            cl_mask = labels == cl
            mk = cluster_markers[cl % len(cluster_markers)]
            for ct in np.unique(snc_ct[cl_mask]):
                both = cl_mask & (snc_ct == ct)
                if both.sum() > 0:
                    ax.scatter(snc_coords[both, 0], snc_coords[both, 1],
                               c=ct_colors.get(ct, '#999'), s=30, alpha=0.9,
                               marker=mk, edgecolors='black', linewidths=0.3,
                               zorder=5, rasterized=True)

        # Cluster rings (dashed circles)
        for cl in unique_labels:
            cl_mask = labels == cl
            cl_coords = snc_coords[cl_mask]
            cx, cy = cl_coords.mean(axis=0)
            max_dist = np.max(np.sqrt((cl_coords[:, 0] - cx)**2 + (cl_coords[:, 1] - cy)**2))
            radius = max(max_dist + 200, 350)
            circle = plt.Circle((cx, cy), radius, fill=False,
                                edgecolor=cluster_pal[cl % len(cluster_pal)],
                                linewidth=1.5, linestyle=(0, (5, 3)),
                                alpha=0.7, zorder=4)
            ax.add_patch(circle)

        format_spatial_ax(ax)
        if col == 0:
            ax.set_ylabel(f'{sc_lab}\n+ SnC', fontsize=5, fontweight='bold', labelpad=4)

        # Overlay row colorbar
        if col == n_samples - 1:
            cax2 = fig.add_subplot(gs[row_overlay, n_samples])
            norm = Normalize(vmin=vmin, vmax=vmax)
            sm = plt.cm.ScalarMappable(cmap='RdYlBu_r', norm=norm)
            sm.set_array([])
            cb2 = fig.colorbar(sm, cax=cax2)
            cb2.ax.tick_params(labelsize=4, length=1.5, width=0.4)
            cb2.outline.set_linewidth(0.4)

# Hide colorbar slot for Row 0
cax_empty = fig.add_subplot(gs[0, n_samples])
cax_empty.axis('off')

# ── Combined legend (bottom center) ──────────────────────────────────────────
handles_all = []

# Section header: Cell Type
handles_all.append(Line2D([0], [0], color='none', marker='None', linestyle='None', label='── Cell Type ──'))
for ct in present_cts:
    handles_all.append(Line2D([0], [0], marker='o', color='w',
                              markerfacecolor=ct_colors.get(ct, '#999'),
                              markersize=4, markeredgewidth=0, label=ct))

# Section header: DBSCAN
handles_all.append(Line2D([0], [0], color='none', marker='None', linestyle='None', label='── DBSCAN ──'))
handles_all.append(Line2D([0], [0], marker='x', color='#888', markersize=4,
                          markeredgewidth=0.6, linestyle='None', label='Noise'))
max_cl = max((max(set(dbscan_labels[sid]) - {-1}, default=-1) for sid in samples), default=-1)
for cl in range(max_cl + 1):
    mk = cluster_markers[cl % len(cluster_markers)]
    handles_all.append(Line2D([0], [0], marker=mk, color='w',
                              markerfacecolor=cluster_pal[cl % len(cluster_pal)],
                              markeredgecolor='black', markeredgewidth=0.3,
                              markersize=5, label=f'Cluster {cl}'))

fig.legend(handles=handles_all, loc='lower center', bbox_to_anchor=(0.48, -0.015),
           frameon=True, fancybox=False, edgecolor='#CCC', fontsize=5,
           ncol=max(len(handles_all) // 2, 6), handletextpad=0.3,
           borderpad=0.5, columnspacing=0.6, labelspacing=0.4)

plt.savefig(str(FIGURES_DIR / 'Fig_multipanel_modules_snc_celltype_clusters.pdf'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_multipanel_modules_snc_celltype_clusters.svg'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_multipanel_modules_snc_celltype_clusters.pdf / .svg")

In [ ]:
adata_sel

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10b-1: BUILD SPATIAL NEIGHBOR GRAPH
# ═══════════════════════════════════════════════════════════════════════════════
#
# GOAL: Construct the spatial connectivity graph for each sample using the
#       actual Visium hexagonal grid topology (k=6 nearest neighbors).
#
# WHY: All downstream spatial statistics (Moran's I, join counts, Gi*, 
#       neighborhood enrichment) require a spatial weights matrix that 
#       defines which spots are "neighbors." 
#
#       Visium spots sit on a hexagonal grid with ~100μm center-to-center 
#       spacing. Each spot has exactly 6 immediate neighbors. Using k=6 
#       nearest neighbors recovers this grid structure.
#
#       This is fundamentally different from DBSCAN/Ripley's L, which treat 
#       spots as freely placed points in continuous space. The grid-based 
#       graph ensures our statistical tests respect the actual data geometry.
#
# OUTPUT: sample_adatas dict — per-sample AnnData objects with 
#         obsp['spatial_connectivities'] (sparse adjacency matrix)
# ═══════════════════════════════════════════════════════════════════════════════

import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("10b-1: BUILD SPATIAL NEIGHBOR GRAPH")
print("="*60)
print()
print("Each Visium spot has ~6 hexagonal neighbors.")
print("We build a k=6 nearest-neighbor graph per sample")
print("to define spatial adjacency for all downstream tests.")
print()

sample_adatas = {}

for sid in samples:
    name, _ = get_meta(sid)
    mask = adata_sel.obs['sample_id'] == sid
    ad = adata_sel[mask].copy()
    
    # Build connectivity graph (k=6 for Visium hexagonal grid)
    sq.gr.spatial_neighbors(ad, n_neighs=6, coord_type='generic')
    sample_adatas[sid] = ad
    
    # Report
    n_spots = ad.n_obs
    n_snc = (ad.obs['is_senescent'] == 1).sum()
    W = ad.obsp['spatial_connectivities']
    n_edges = W.nnz // 2  # symmetric, so divide by 2
    avg_neighbors = W.sum(axis=1).A1.mean() if sparse.issparse(W) else W.sum(axis=1).mean()
    
    print(f"  {name}:")
    print(f"    Spots: {n_spots}  |  SnC: {n_snc} ({n_snc/n_spots*100:.1f}%)")
    print(f"    Edges: {n_edges}  |  Avg neighbors/spot: {avg_neighbors:.1f}")
    print()

print("✓ Spatial neighbor graphs built for all samples.")
print("  Stored in: sample_adatas[sid].obsp['spatial_connectivities']")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10b-2: MORAN'S I — GLOBAL SPATIAL AUTOCORRELATION
# ═══════════════════════════════════════════════════════════════════════════════
#
# GOAL: Test whether senescence labels and scores are spatially autocorrelated.
#
# WHAT IT DOES:
#   Moran's I measures correlation between a spot's value and the average 
#   of its neighbors' values on the Visium grid.
#     I > 0 → clustered    I ≈ 0 → random    I < 0 → dispersed
#
#   The Moran scatter plot (x = value, y = spatial lag) is the standard 
#   visualization. The slope of the regression line = Moran's I.
#   Quadrants: HH = hot spot, LL = cold spot, HL/LH = spatial outlier.
#
# OUTPUT: df_morans, Moran scatter plots, summary bar chart
# ═══════════════════════════════════════════════════════════════════════════════

print("="*60)
print("10b-2: MORAN'S I — GLOBAL SPATIAL AUTOCORRELATION")
print("="*60)
print()
print("Question: Are SnC spots / high senescence scores spatially clustered?")
print("Method:   Moran's I with 999 permutations on the Visium grid graph")
print()

score_cols_test = [
    'is_senescent',
    'sen_score',
    'Score_p53_Targets',
    'Score_CellCycleArrest',
    'Score_SASP',
    'Score_AntiApoptosis',
    'Score_DDR',
    'Score_CellSurfaceMarkers',
    'Score_LysosomalContent',
    'Score_SD_TMC',
    'Score_SenMayo',
    'Score_Fridman_Up',
]

morans_results = []

for sid in samples:
    name, _ = get_meta(sid)
    ad = sample_adatas[sid]
    
    ad.obs['is_senescent_num'] = ad.obs['is_senescent'].astype(float)
    
    test_cols = ['is_senescent_num'] + [c for c in score_cols_test[1:] if c in ad.obs.columns]
    
    sq.gr.spatial_autocorr(
        ad, mode='moran', genes=test_cols, attr='obs', n_perms=999,
    )
    
    print(f"  {name}:")
    print(f"  {'Variable':<26s} {'I':>8s} {'p':>8s} {'':>4s} {'':>10s}")
    print(f"  {'─'*60}")
    
    for col in test_cols:
        display_name = col.replace('is_senescent_num', 'SnC (binary)')
        display_name = display_name.replace('sen_score', 'Senescence Score')
        display_name = display_name.replace('Score_', '')
        
        I_val = ad.uns['moranI'].loc[col, 'I']
        pval = ad.uns['moranI'].loc[col, 'pval_sim']
        
        sig = ''
        if pval < 0.001: sig = '***'
        elif pval < 0.01: sig = '**'
        elif pval < 0.05: sig = '*'
        
        interp = 'CLUSTERED' if (I_val > 0 and pval < 0.05) else ('dispersed' if (I_val < 0 and pval < 0.05) else 'random')
        
        print(f"  {display_name:<26s} {I_val:>+8.4f} {pval:>8.4f} {sig:>4s} {interp:>10s}")
        
        morans_results.append({
            'sample': name, 'variable': display_name, 'variable_raw': col,
            'morans_I': I_val, 'p_value': pval, 'significant': sig,
            'interpretation': interp,
        })
    print()

df_morans = pd.DataFrame(morans_results)

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: MORAN SCATTER PLOTS — sen_score (the key variable)
# ══════════════════════════════════════════════════════════════════════════════
# x = standardized score, y = spatial lag (avg of neighbors)
# Slope of regression line = Moran's I

from scipy import sparse as sp_sparse

fig, axes = plt.subplots(1, n_samples, figsize=(7.2, 1.8))
if n_samples == 1:
    axes = [axes]

for col, sid in enumerate(samples):
    name, _ = get_meta(sid)
    ad = sample_adatas[sid]
    ax = axes[col]
    
    # Use sen_score
    score_col = 'sen_score' if 'sen_score' in ad.obs.columns else 'is_senescent_num'
    x = ad.obs[score_col].values.astype(float)
    
    # Standardize
    x_std = (x - x.mean()) / x.std()
    
    # Spatial lag: W @ x_std (row-normalized weights)
    W = ad.obsp['spatial_connectivities']
    if sp_sparse.issparse(W):
        row_sums = np.array(W.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1
        W_norm = W.multiply(1 / row_sums[:, np.newaxis])
        lag = np.array(W_norm @ x_std).flatten()
    else:
        row_sums = W.sum(axis=1)
        row_sums[row_sums == 0] = 1
        W_norm = W / row_sums[:, np.newaxis]
        lag = W_norm @ x_std
    
    # Get Moran's I and p-value
    moran_row = df_morans[(df_morans['sample'] == name) & 
                          (df_morans['variable'].str.contains('Senescence Score|SnC'))].iloc[0]
    I_val = moran_row['morans_I']
    pval = moran_row['p_value']
    
    # Color by quadrant
    is_snc = ad.obs['is_senescent'].values == 1
    colors = np.where(is_snc, '#D62728', '#CCCCCC')
    
    ax.scatter(x_std[~is_snc], lag[~is_snc], c='#CCCCCC', s=1, alpha=0.3,
               edgecolors='none', rasterized=True)
    ax.scatter(x_std[is_snc], lag[is_snc], c='#D62728', s=6, alpha=0.7,
               edgecolors='none', rasterized=True, zorder=3)
    
    # Regression line
    m, b = np.polyfit(x_std, lag, 1)
    x_line = np.linspace(x_std.min(), x_std.max(), 100)
    ax.plot(x_line, m * x_line + b, color='#333333', linewidth=1, linestyle='-')
    
    # Quadrant lines
    ax.axhline(0, color='#999999', linewidth=0.4, linestyle='-')
    ax.axvline(0, color='#999999', linewidth=0.4, linestyle='-')
    
    # Stats in top-left
    sig = moran_row['significant']
    ax.text(0.04, 0.96, f"I = {I_val:+.3f}\np = {pval:.3f} {sig}",
            transform=ax.transAxes, fontsize=5.5, fontweight='bold',
            va='top', ha='left', family='monospace',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                      edgecolor='#CCCCCC', linewidth=0.4, alpha=0.9))
    
    # Quadrant labels (subtle)
    ax.text(0.97, 0.97, 'HH', transform=ax.transAxes, fontsize=4.5,
            color='#999999', va='top', ha='right')
    ax.text(0.03, 0.03, 'LL', transform=ax.transAxes, fontsize=4.5,
            color='#999999', va='bottom', ha='left')
    ax.text(0.97, 0.03, 'HL', transform=ax.transAxes, fontsize=4.5,
            color='#999999', va='bottom', ha='right')
    ax.text(0.03, 0.97, 'LH', transform=ax.transAxes, fontsize=4.5,
            color='#999999', va='top', ha='left', 
            bbox=dict(facecolor='none', edgecolor='none'))  # avoid overlap with stats
    
    ax.set_title(name, fontsize=7, fontweight='bold', pad=3)
    if col == 0:
        ax.set_ylabel('Spatial lag', fontsize=6)
        ax.set_xlabel('Senescence score (std)', fontsize=6)
    else:
        ax.set_yticklabels([])
    ax.tick_params(labelsize=5)

# Minimal legend
handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#D62728',
           markersize=3, markeredgewidth=0, label='SnC'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#CCCCCC',
           markersize=3, markeredgewidth=0, label='Non-SnC'),
]
axes[-1].legend(handles=handles, loc='lower right', fontsize=4.5,
                frameon=True, fancybox=False, edgecolor='#CCC',
                handletextpad=0.2, borderpad=0.2)

plt.tight_layout(w_pad=0.3)
plt.savefig(str(FIGURES_DIR / 'Fig_moran_scatter.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_moran_scatter.svg'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_moran_scatter.pdf / .svg")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: MORAN'S I SUMMARY HEATMAP (compact, all scores × all samples)
# ══════════════════════════════════════════════════════════════════════════════

pivot_I = df_morans.pivot(index='variable', columns='sample', values='morans_I')
pivot_p = df_morans.pivot(index='variable', columns='sample', values='p_value')

sample_order = [SELECTED[sid]['name'] for sid in samples]
pivot_I = pivot_I[sample_order]
pivot_p = pivot_p[sample_order]

fig, ax = plt.subplots(figsize=(3.2, 3.8))

vmax = max(abs(pivot_I.values.min()), abs(pivot_I.values.max()), 0.1)
im = ax.imshow(pivot_I.values, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)

for i in range(pivot_I.shape[0]):
    for j in range(pivot_I.shape[1]):
        val = pivot_I.values[i, j]
        pv = pivot_p.values[i, j]
        sig = ''
        if pv < 0.001: sig = '***'
        elif pv < 0.01: sig = '**'
        elif pv < 0.05: sig = '*'
        
        txt_color = 'white' if abs(val) > vmax * 0.55 else 'black'
        ax.text(j, i, f'{val:+.3f}', ha='center', va='center',
                fontsize=5, color=txt_color)
        if sig:
            ax.text(j, i + 0.32, sig, ha='center', va='center',
                    fontsize=4.5, color=txt_color, fontweight='bold')

ax.set_xticks(range(len(sample_order)))
ax.set_xticklabels(sample_order, fontsize=6)
ax.set_yticks(range(len(pivot_I.index)))
ax.set_yticklabels(pivot_I.index, fontsize=5.5)
ax.set_title("Moran's I", fontsize=8, fontweight='bold', pad=6)

cbar = plt.colorbar(im, ax=ax, shrink=0.5, aspect=20, pad=0.04)
cbar.ax.tick_params(labelsize=5)
cbar.set_label("Moran's I", fontsize=5.5)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'Fig_morans_I_heatmap.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_morans_I_heatmap.svg'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_morans_I_heatmap.pdf / .svg")

df_morans.to_csv(RESULTS_DIR / 'spatial_morans_I.csv', index=False)
print(f"✓ Saved: {RESULTS_DIR / 'spatial_morans_I.csv'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10b-3: CELL-TYPE-STRATIFIED BIVARIATE MORAN'S I
# ═══════════════════════════════════════════════════════════════════════════════
# Env: omicverse
# Question: Within each cell type, is senescence score spatially
#           associated with SASP/SenMayo/other module scores?
# Method:   Subset to spots of one cell type → rebuild spatial graph →
#           Bivariate Moran's I (sen_score × module score)
# Min spots: 50 per cell type per sample
# ═══════════════════════════════════════════════════════════════════════════════

import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from esda.moran import Moran_BV
from libpysal.weights import W as LibW
from scipy import sparse as sp_sparse
from scipy.spatial import KDTree
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("10b-9: CELL-TYPE-STRATIFIED BIVARIATE MORAN'S I")
print("=" * 60)
print()
print("Question: Within each cell type, is senescence score spatially")
print("          associated with SASP/SenMayo/other module scores?")
print("Method:   Per cell type subset → rebuild spatial neighbors →")
print("          Bivariate Moran's I (sen_score × module), 999 permutations")
print()

# ── Config ───────────────────────────────────────────────────────────────────
MIN_SPOTS = 50

module_cols = [
    ('Score_p53_Targets',      'p53_Targets'),
    ('Score_CellCycleArrest',  'CellCycleArrest'),
    ('Score_SASP',             'SASP'),
    ('Score_AntiApoptosis',    'AntiApoptosis'),
    ('Score_DDR',              'DDR'),
    ('Score_CellSurfaceMarkers','CellSurfaceMarkers'),
    ('Score_LysosomalContent', 'LysosomalContent'),
    ('Score_SD_TMC',           'SD_TMC'),
    ('Score_SenMayo',          'SenMayo'),
    ('Score_Fridman_Up',       'Fridman_Up'),
]

# ── Discover all cell types with enough spots in at least 1 sample ───────────
all_ct_counts = {}
for sid in samples:
    name, group = get_meta(sid)
    ad = sample_adatas[sid]
    cts = ad.obs['dominant_celltype'].value_counts()
    for ct, n in cts.items():
        if ct not in all_ct_counts:
            all_ct_counts[ct] = {}
        all_ct_counts[ct][name] = n

print("  Cell type spot counts per sample:")
print(f"  {'Cell Type':<20s}", end='')
for sid in samples:
    name, _ = get_meta(sid)
    print(f" {name:>8s}", end='')
print(f" {'Include?':>10s}")
print(f"  {'─'*(20 + 8*len(samples) + 12)}")

CELL_TYPES_TEST = []
for ct in sorted(all_ct_counts.keys()):
    counts = [all_ct_counts[ct].get(SELECTED[sid]['name'], 0) for sid in samples]
    n_pass = sum(1 for c in counts if c >= MIN_SPOTS)
    include = n_pass >= 2  # need at least 2 samples with enough spots
    if include:
        CELL_TYPES_TEST.append(ct)
    print(f"  {ct:<20s}", end='')
    for c in counts:
        marker = '✓' if c >= MIN_SPOTS else ''
        print(f" {c:>6d}{marker:>2s}", end='')
    print(f" {'YES' if include else 'no':>10s}")

print(f"\n  Testing: {CELL_TYPES_TEST}")

# ── Helper: build spatial graph from coordinates ─────────────────────────────
def build_spatial_w(coords, k=6):
    """Build k-nearest-neighbor spatial weights from coordinates."""
    tree = KDTree(coords)
    dists, indices = tree.query(coords, k=k+1)
    
    neighbors = {}
    weights = {}
    for i in range(len(coords)):
        nbrs = indices[i, 1:].tolist()
        wts = [1.0] * len(nbrs)
        neighbors[i] = nbrs
        weights[i] = wts
    
    return LibW(neighbors, weights)

# ── Run per cell type per sample ─────────────────────────────────────────────
ct_biv_results = []

for sid in samples:
    name, group = get_meta(sid)
    ad = sample_adatas[sid]
    
    print(f"\n  {name} ({group}):")
    
    for ct in CELL_TYPES_TEST:
        ct_mask = ad.obs['dominant_celltype'] == ct
        n_ct = ct_mask.sum()
        
        if n_ct < MIN_SPOTS:
            print(f"    {ct}: SKIP (n={n_ct} < {MIN_SPOTS})")
            continue
        
        ad_ct = ad[ct_mask].copy()
        coords = ad_ct.obsm['spatial']
        w = build_spatial_w(coords, k=6)
        
        x_sen = ad_ct.obs['sen_score'].astype(float).values
        n_snc = (ad_ct.obs['is_senescent'] == 1).sum()
        
        print(f"    {ct}: n={n_ct}, SnC={n_snc} ({n_snc/n_ct*100:.1f}%)")
        print(f"      {'Module':<22s} {'Biv_I':>8s} {'p':>8s} {'':>4s} {'Interpretation':<28s}")
        print(f"      {'─'*74}")
        
        for col, display_name in module_cols:
            if col not in ad_ct.obs.columns:
                continue
            
            y_score = ad_ct.obs[col].astype(float).values
            
            if x_sen.std() < 1e-10 or y_score.std() < 1e-10:
                print(f"      {display_name:<22s} {'N/A':>8s} {'N/A':>8s} {'':>4s} {'constant score':<28s}")
                ct_biv_results.append({
                    'sample': name, 'group': group, 'cell_type': ct,
                    'n_spots': n_ct, 'n_snc': n_snc,
                    'module': display_name, 'module_raw': col,
                    'bivariate_I': np.nan, 'p_value': np.nan,
                    'significant': '', 'interpretation': 'constant score',
                })
                continue
            
            bv = Moran_BV(x_sen, y_score, w, permutations=999)
            I_val = bv.I
            pval = bv.p_sim
            
            sig = ''
            if pval < 0.001: sig = '***'
            elif pval < 0.01: sig = '**'
            elif pval < 0.05: sig = '*'
            
            if I_val > 0 and pval < 0.05:
                interp = 'HIGH sen near HIGH module'
            elif I_val < 0 and pval < 0.05:
                interp = 'HIGH sen near LOW module'
            else:
                interp = 'no spatial association'
            
            print(f"      {display_name:<22s} {I_val:>+8.4f} {pval:>8.4f} {sig:>4s} {interp:<28s}")
            
            ct_biv_results.append({
                'sample': name, 'group': group, 'cell_type': ct,
                'n_spots': n_ct, 'n_snc': n_snc,
                'module': display_name, 'module_raw': col,
                'bivariate_I': I_val, 'p_value': pval,
                'significant': sig, 'interpretation': interp,
            })
        
        print()

df_ct_biv = pd.DataFrame(ct_biv_results)

# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 1: HEATMAP PER CELL TYPE — one subplot per cell type
# ═════════════════════════════════════════════════════════════════════════════

print(f"{'='*60}")
print("FIGURE: Cell-Type-Stratified Bivariate Moran's I Heatmaps")
print(f"{'='*60}")

sample_order = [SELECTED[sid]['name'] for sid in samples]
cts_with_data = [ct for ct in CELL_TYPES_TEST if ct in df_ct_biv['cell_type'].values]
n_cts = len(cts_with_data)

if n_cts == 0:
    print("  No cell types with sufficient data")
else:
    # Horizontal layout — 1 row × n_cts columns, compact
    col_width = 0.45 + 0.55 * max(len(sample_order) for _ in cts_with_data)
    fig_width = col_width * n_cts + 2.0
    fig_height = 4.5
    fig, axes = plt.subplots(1, n_cts, figsize=(fig_width, fig_height),
                             gridspec_kw={'wspace': 0.08})
    if n_cts == 1:
        axes = [axes]
    
    for ct_idx, ct in enumerate(cts_with_data):
        ax = axes[ct_idx]
        sub = df_ct_biv[df_ct_biv['cell_type'] == ct]
        
        if sub.empty:
            ax.set_title(f"{ct} (no data)", fontsize=8)
            ax.axis('off')
            continue
        
        pivot_I = sub.pivot(index='module', columns='sample', values='bivariate_I')
        pivot_p = sub.pivot(index='module', columns='sample', values='p_value')
        
        avail_samples = [s for s in sample_order if s in pivot_I.columns]
        if not avail_samples:
            ax.set_title(f"{ct} (no samples)", fontsize=8)
            ax.axis('off')
            continue
        
        pivot_I = pivot_I[avail_samples]
        pivot_p = pivot_p[avail_samples]
        
        vmax = max(abs(np.nanmin(pivot_I.values)), abs(np.nanmax(pivot_I.values)), 0.05)
        im = ax.imshow(pivot_I.values, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)
        
        for i in range(pivot_I.shape[0]):
            for j in range(pivot_I.shape[1]):
                val = pivot_I.values[i, j]
                pv = pivot_p.values[i, j]
                if np.isnan(val):
                    ax.text(j, i, 'N/A', ha='center', va='center', fontsize=7, color='gray')
                    continue
                sig = '***' if pv < 0.001 else ('**' if pv < 0.01 else ('*' if pv < 0.05 else ''))
                txt_color = 'white' if abs(val) > vmax * 0.55 else 'black'
                ax.text(j, i, f'{val:+.3f}', ha='center', va='center', fontsize=7, color=txt_color)
                if sig:
                    ax.text(j, i + 0.32, sig, ha='center', va='center', fontsize=5.5,
                            color=txt_color, fontweight='bold')
        
        # X labels on all heatmaps — rotated 45
        ax.set_xticks(range(len(avail_samples)))
        ax.set_xticklabels(avail_samples, fontsize=8, rotation=45, ha='right')
        
        # Y labels only on first column
        ax.set_yticks(range(len(pivot_I.index)))
        if ct_idx == 0:
            ax.set_yticklabels(pivot_I.index, fontsize=8)
        else:
            ax.set_yticklabels([])
        
        # Title — cell type name only
        ax.set_title(f"{ct}", fontsize=9, fontweight='bold', pad=4)
    
    # Single colorbar on the right
    cbar = fig.colorbar(im, ax=axes, shrink=0.35, aspect=15, pad=0.03)
    cbar.ax.tick_params(labelsize=7)
    cbar.set_label("Bivariate I", fontsize=8)
    
    fig.suptitle("Cell-Type-Stratified Bivariate Moran's I\n(Senescence Score × Module Scores)",
                 fontsize=10, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / 'Fig_biv_moran_by_celltype.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(str(FIGURES_DIR / 'Fig_biv_moran_by_celltype.svg'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print("✓ Fig_biv_moran_by_celltype.pdf / .svg")

# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 2: COMPARISON — Whole tissue vs Cell-type-stratified (key modules)
# ═════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print("FIGURE 2: Whole Tissue vs Stratified Comparison")
print(f"{'='*60}")

key_modules = ['SASP', 'SenMayo', 'LysosomalContent', 'Fridman_Up']

prev_file = RESULTS_DIR / 'spatial_biv_moran_senscore.csv'
if prev_file.exists():
    df_whole = pd.read_csv(prev_file)
    
    categories = ['Whole tissue'] + cts_with_data
    
    ct_colors = {
        'Whole tissue': '#333333',
        'Excitatory': '#2ca02c',
        'Inhibitory': '#ff7f0e',
        'Oligodendrocyte': '#9467bd',
        'Astrocyte': '#1f77b4',
        'Microglia': '#8c564b',
        'OPC': '#e377c2',
        'Endothelial': '#d62728',
        'Pericyte': '#17becf',
        'PVM': '#bcbd22',
        'VLMC': '#f7b6d2',
        'VSMC': '#7f7f7f',
        'Adaptive': '#aec7e8',
        'Vascular': '#17becf',
    }
    
    fig, axes = plt.subplots(1, len(key_modules), figsize=(4.0 * len(key_modules), 3.5))
    if len(key_modules) == 1:
        axes = [axes]
    
    for ax_idx, mod in enumerate(key_modules):
        ax = axes[ax_idx]
        
        x = np.arange(len(sample_order))
        width = 0.75 / len(categories)
        
        for cat_idx, cat in enumerate(categories):
            vals = []
            pvals = []
            for sname in sample_order:
                if cat == 'Whole tissue':
                    row = df_whole[(df_whole['sample'] == sname) & (df_whole['module'] == mod)]
                else:
                    row = df_ct_biv[(df_ct_biv['sample'] == sname) &
                                    (df_ct_biv['module'] == mod) &
                                    (df_ct_biv['cell_type'] == cat)]
                
                if len(row) > 0 and not np.isnan(row.iloc[0]['bivariate_I']):
                    vals.append(row.iloc[0]['bivariate_I'])
                    pvals.append(row.iloc[0]['p_value'])
                else:
                    vals.append(0)
                    pvals.append(1)
            
            bars = ax.bar(x + cat_idx * width, vals, width,
                         label=cat, color=ct_colors.get(cat, '#999'), alpha=0.8)
            
            for j, (v, p) in enumerate(zip(vals, pvals)):
                if p < 0.001: star = '***'
                elif p < 0.01: star = '**'
                elif p < 0.05: star = '*'
                else: star = ''
                if star:
                    y_pos = v + 0.003 if v >= 0 else v - 0.01
                    ax.text(x[j] + cat_idx * width, y_pos, star,
                           ha='center', va='bottom', fontsize=3.5)
        
        ax.axhline(0, color='black', linewidth=0.5)
        ax.set_xticks(x + width * (len(categories) - 1) / 2)
        ax.set_xticklabels(sample_order, fontsize=6, rotation=45, ha='right')
        ax.set_title(mod, fontsize=8, fontweight='bold')
        ax.set_ylabel("Bivariate I", fontsize=6)
        ax.tick_params(labelsize=5)
        
        if ax_idx == 0:
            ax.legend(fontsize=4, frameon=True, fancybox=False, edgecolor='#CCCCCC',
                     handletextpad=0.3, borderpad=0.3, loc='best')
    
    fig.suptitle("Whole Tissue vs Cell-Type-Stratified\nBivariate Moran's I (sen_score × modules)",
                 fontsize=9, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / 'Fig_biv_moran_stratified_comparison.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(str(FIGURES_DIR / 'Fig_biv_moran_stratified_comparison.svg'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print("✓ Fig_biv_moran_stratified_comparison.pdf / .svg")
else:
    print("  (No whole-tissue results found — run Cell 10b-8 first)")

# ── Save ─────────────────────────────────────────────────────────────────────
df_ct_biv.to_csv(RESULTS_DIR / 'spatial_biv_moran_by_celltype.csv', index=False)
print(f"\n✓ Saved: {RESULTS_DIR / 'spatial_biv_moran_by_celltype.csv'}")

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("SUMMARY — CELL-TYPE-STRATIFIED RESULTS")
print(f"{'='*60}")

for ct in cts_with_data:
    sub = df_ct_biv[df_ct_biv['cell_type'] == ct]
    print(f"\n  {ct}:")
    for mod in ['SASP', 'SenMayo', 'LysosomalContent', 'Fridman_Up']:
        mod_sub = sub[sub['module'] == mod]
        if mod_sub.empty:
            continue
        n_pos = ((mod_sub['bivariate_I'] > 0) & (mod_sub['p_value'] < 0.05)).sum()
        n_neg = ((mod_sub['bivariate_I'] < 0) & (mod_sub['p_value'] < 0.05)).sum()
        n_ns = (mod_sub['p_value'] >= 0.05).sum()
        n_nan = mod_sub['bivariate_I'].isna().sum()
        mean_I = mod_sub['bivariate_I'].mean()
        print(f"    {mod:<22s}: mean I={mean_I:+.4f} | +sig={n_pos}, -sig={n_neg}, ns={n_ns} / {len(mod_sub)} samples")

print(f"\n  Key question: Does SASP direction change within cell types?")
for ct in cts_with_data:
    sasp_sub = df_ct_biv[(df_ct_biv['cell_type'] == ct) & (df_ct_biv['module'] == 'SASP')]
    if sasp_sub.empty:
        continue
    valid = sasp_sub['bivariate_I'].dropna()
    if len(valid) == 0:
        continue
    mean_I = valid.mean()
    direction = "POSITIVE (co-localize)" if mean_I > 0 else "NEGATIVE (segregate)"
    print(f"    {ct}: SASP mean I = {mean_I:+.4f} → {direction}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10b-3: JOIN COUNT STATISTICS — SnC-SnC NEIGHBOR CO-OCCURRENCE
# ═══════════════════════════════════════════════════════════════════════════════
#
# GOAL: Do SnC spots neighbor other SnC spots more than expected by chance?
# METHOD: Count SnC-SnC edges on grid, compare to 9999 permutations
# ═══════════════════════════════════════════════════════════════════════════════

print("="*60)
print("10b-3: JOIN COUNT STATISTICS")
print("="*60)
print()
print("Question: Do SnC spots neighbor other SnC spots more than expected?")
print("Method:   Count SnC-SnC edges, compare to 9999 label permutations")
print()

n_permutations = 9999
join_count_results = []
perm_distributions = {}

for sid in samples:
    name, _ = get_meta(sid)
    ad = sample_adatas[sid]
    
    is_snc = (ad.obs['is_senescent'] == 1).values.astype(int)
    W = ad.obsp['spatial_connectivities']
    
    if sparse.issparse(W):
        W_dense = W.toarray()
    else:
        W_dense = np.array(W)
    
    W_upper = np.triu(W_dense, k=1)
    total_edges = int(W_upper.sum())
    n_snc = is_snc.sum()
    n_total = len(is_snc)
    
    observed_joins = int((W_upper * np.outer(is_snc, is_snc)).sum())
    
    perm_joins = np.zeros(n_permutations)
    for p in range(n_permutations):
        perm_labels = np.random.permutation(is_snc)
        perm_joins[p] = (W_upper * np.outer(perm_labels, perm_labels)).sum()
    
    expected = np.mean(perm_joins)
    std_perm = np.std(perm_joins)
    z_score = (observed_joins - expected) / std_perm if std_perm > 0 else 0
    p_value = np.mean(perm_joins >= observed_joins)
    fold_change = observed_joins / expected if expected > 0 else np.inf
    
    sig = ''
    if p_value < 0.001: sig = '***'
    elif p_value < 0.01: sig = '**'
    elif p_value < 0.05: sig = '*'
    
    print(f"  {name}:")
    print(f"    {n_snc} SnC / {n_total} spots  |  {total_edges} edges")
    print(f"    Observed: {observed_joins}  Expected: {expected:.1f}  FC: {fold_change:.2f}×")
    print(f"    z = {z_score:+.2f}  p = {p_value:.4f} {sig}")
    print(f"    → {'SIGNIFICANTLY CO-LOCALIZED' if p_value < 0.05 else 'Not significant'}")
    print()
    
    join_count_results.append({
        'sample': name, 'n_spots': n_total, 'n_snc': n_snc,
        'n_edges': total_edges, 'observed_joins': observed_joins,
        'expected_joins': expected, 'fold_change': fold_change,
        'z_score': z_score, 'p_value': p_value, 'significant': sig,
    })
    perm_distributions[name] = perm_joins

df_joins = pd.DataFrame(join_count_results)

# ── Figure: Permutation distributions (compact) ──────────────────────────────
fig, axes = plt.subplots(1, n_samples, figsize=(7.2, 1.6))
if n_samples == 1:
    axes = [axes]

for col, sid in enumerate(samples):
    name = SELECTED[sid]['name']
    row = df_joins[df_joins['sample'] == name].iloc[0]
    perm_vals = perm_distributions[name]
    
    ax = axes[col]
    ax.hist(perm_vals, bins=30, color='#D9D9D9', edgecolor='white',
            linewidth=0.3, density=True)
    ax.axvline(row['observed_joins'], color='#D62728', linewidth=1.2, linestyle='-')
    ax.axvline(row['expected_joins'], color='#333333', linewidth=0.6, linestyle=':')
    
    # Stats top-left
    ax.text(0.04, 0.96,
            f"obs = {int(row['observed_joins'])}\nexp = {row['expected_joins']:.0f}\n"
            f"z = {row['z_score']:+.2f}\np = {row['p_value']:.4f} {row['significant']}",
            transform=ax.transAxes, fontsize=4.5, va='top', ha='left',
            family='monospace',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                      edgecolor='#CCC', linewidth=0.3, alpha=0.9))
    
    ax.set_title(name, fontsize=7, fontweight='bold', pad=3)
    if col == 0:
        ax.set_ylabel('Density', fontsize=6)
    else:
        ax.set_yticklabels([])
    ax.set_xlabel('SnC-SnC joins', fontsize=5.5)
    ax.tick_params(labelsize=5)

# Minimal legend on last panel
handles = [
    Line2D([0], [0], color='#D62728', linewidth=1.2, label='Observed'),
    Line2D([0], [0], color='#333333', linewidth=0.6, linestyle=':', label='Expected'),
]
axes[-1].legend(handles=handles, loc='upper right', fontsize=4.5,
                frameon=True, fancybox=False, edgecolor='#CCC',
                handletextpad=0.3, borderpad=0.2)

plt.tight_layout(w_pad=0.3)
plt.savefig(str(FIGURES_DIR / 'Fig_join_counts.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_join_counts.svg'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_join_counts.pdf / .svg")

df_joins.to_csv(RESULTS_DIR / 'spatial_join_counts.csv', index=False)
print(f"✓ Saved: {RESULTS_DIR / 'spatial_join_counts.csv'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10b-4: GETIS-ORD Gi* — LOCAL HOT SPOT DETECTION
# ═══════════════════════════════════════════════════════════════════════════════
#
# GOAL: Identify which spots are in significant senescence hot/cold spots.
#       Replaces DBSCAN with grid-appropriate, per-spot p-values.
#
# Gi* z-score per spot:
#   z > +1.96  → hot spot (p<0.05)
#   z > +2.576 → hot spot (p<0.01)
#   z < -1.96  → cold spot
# ═══════════════════════════════════════════════════════════════════════════════

print("="*60)
print("10b-4: GETIS-ORD Gi* — LOCAL HOT SPOT DETECTION")
print("="*60)
print()
print("Question: Where are the senescence hot spots on the tissue?")
print("Method:   Per-spot Gi* z-score on the grid neighbor graph")
print()

def getis_ord_gi_star(values, W):
    n = len(values)
    x = np.asarray(values, dtype=float)
    x_mean = x.mean()
    S = x.std()
    if S == 0:
        return np.zeros(n)
    if sparse.issparse(W):
        W_arr = W.toarray()
    else:
        W_arr = np.array(W)
    np.fill_diagonal(W_arr, 1)
    Wi = W_arr.sum(axis=1)
    Wi2 = (W_arr ** 2).sum(axis=1)
    num = W_arr @ x - x_mean * Wi
    denom = S * np.sqrt((n * Wi2 - Wi**2) / (n - 1))
    denom[denom == 0] = np.nan
    return num / denom

hotspot_summary = []

for sid in samples:
    name, _ = get_meta(sid)
    ad = sample_adatas[sid]
    W = ad.obsp['spatial_connectivities']
    
    score_col = 'sen_score' if 'sen_score' in ad.obs.columns else 'is_senescent_num'
    gi_star = getis_ord_gi_star(ad.obs[score_col].values, W)
    
    ad.obs['gi_star_sen'] = gi_star
    ad.obs['gi_star_pval'] = 2 * (1 - norm.cdf(np.abs(gi_star)))
    ad.obs['hotspot'] = 'ns'
    ad.obs.loc[gi_star > 1.96, 'hotspot'] = 'hot*'
    ad.obs.loc[gi_star > 2.576, 'hotspot'] = 'hot**'
    ad.obs.loc[gi_star > 3.291, 'hotspot'] = 'hot***'
    ad.obs.loc[gi_star < -1.96, 'hotspot'] = 'cold*'
    ad.obs.loc[gi_star < -2.576, 'hotspot'] = 'cold**'
    
    is_snc = (ad.obs['is_senescent'] == 1).values
    n_snc = is_snc.sum()
    n_hot = (gi_star > 1.96).sum()
    n_hot01 = (gi_star > 2.576).sum()
    n_cold = (gi_star < -1.96).sum()
    n_snc_hot = ((gi_star > 1.96) & is_snc).sum()
    
    print(f"  {name}:")
    print(f"    Hot:  {n_hot} (p<.05)  {n_hot01} (p<.01)  |  Cold: {n_cold}")
    print(f"    SnC in hot spots: {n_snc_hot}/{n_snc} ({n_snc_hot/n_snc*100:.0f}%)")
    print()
    
    hotspot_summary.append({
        'sample': name, 'score_used': score_col,
        'n_spots': ad.n_obs, 'n_snc': n_snc,
        'n_hot_p05': n_hot, 'n_hot_p01': n_hot01, 'n_cold_p05': n_cold,
        'n_snc_in_hot': n_snc_hot,
        'pct_snc_in_hot': n_snc_hot / n_snc * 100 if n_snc > 0 else 0,
    })

df_hotspot = pd.DataFrame(hotspot_summary)

# ── Figure: 3 rows × n_samples ───────────────────────────────────────────────
# Row 0: Gi* z-score continuous
# Row 1: Hot/cold classified + SnC overlay
# Row 2: SnC colored by hot/cold status

fig = plt.figure(figsize=(7.2, 5.0))
gs = gridspec.GridSpec(3, n_samples + 1,
                       width_ratios=[1]*n_samples + [0.03],
                       hspace=0.14, wspace=0.06,
                       left=0.02, right=0.95, top=0.95, bottom=0.03)

for col, sid in enumerate(samples):
    ad = sample_adatas[sid]
    d = sample_data[sid]
    coords = d['coords']
    gi = ad.obs['gi_star_sen'].values
    is_snc = d['is_snc']
    name, _ = get_meta(sid)
    
    vabs = max(abs(np.nanpercentile(gi, 1)), abs(np.nanpercentile(gi, 99)), 3)
    
    # ── Row 0: Gi* continuous ────────────────────────────────────────────────
    ax = fig.add_subplot(gs[0, col])
    sc = ax.scatter(coords[:, 0], coords[:, 1], c=gi, cmap='RdBu_r',
                    s=3, alpha=0.85, vmin=-vabs, vmax=vabs,
                    edgecolors='none', rasterized=True)
    format_spatial_ax(ax)
    ax.set_title(name, fontsize=7, fontweight='bold', pad=3)
    if col == 0:
        ax.set_ylabel('Gi* z-score', fontsize=6, fontweight='bold', labelpad=4)
    
    if col == n_samples - 1:
        cax = fig.add_subplot(gs[0, n_samples])
        cb = fig.colorbar(sc, cax=cax)
        cb.ax.tick_params(labelsize=4, length=1.5, width=0.4)
        cb.outline.set_linewidth(0.4)
    
    # ── Row 1: Hot/cold + SnC ────────────────────────────────────────────────
    ax = fig.add_subplot(gs[1, col])
    
    ns_mask = np.abs(gi) <= 1.96
    ax.scatter(coords[ns_mask, 0], coords[ns_mask, 1], c='#E8E8E8', s=2,
               alpha=0.4, edgecolors='none', rasterized=True)
    
    cold_mask = gi < -1.96
    if cold_mask.sum() > 0:
        ax.scatter(coords[cold_mask, 0], coords[cold_mask, 1], c='#4575B4', s=3,
                   alpha=0.7, edgecolors='none', rasterized=True)
    
    hot_mask = gi > 1.96
    if hot_mask.sum() > 0:
        ax.scatter(coords[hot_mask, 0], coords[hot_mask, 1], c='#D73027', s=3,
                   alpha=0.7, edgecolors='none', rasterized=True)
    
    ax.scatter(d['snc_coords'][:, 0], d['snc_coords'][:, 1], c='black',
               s=10, alpha=0.9, marker='*', linewidths=0, zorder=5, rasterized=True)
    
    format_spatial_ax(ax)
    n_snc_hot = ((gi > 1.96) & is_snc).sum()
    ax.text(0.04, 0.96, f"{n_snc_hot}/{is_snc.sum()} SnC\nin hot spots",
            transform=ax.transAxes, fontsize=4.5, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                      edgecolor='#CCC', linewidth=0.3, alpha=0.9))
    if col == 0:
        ax.set_ylabel('Hot/Cold', fontsize=6, fontweight='bold', labelpad=4)
    
    # ── Row 2: SnC only, colored by whether in hot spot ──────────────────────
    ax = fig.add_subplot(gs[2, col])
    ax.scatter(coords[:, 0], coords[:, 1], c='#F0F0F0', s=1, alpha=0.2,
               edgecolors='none', rasterized=True)
    
    snc_gi = gi[is_snc]
    snc_in_hot = snc_gi > 1.96
    snc_in_cold = snc_gi < -1.96
    snc_ns = ~snc_in_hot & ~snc_in_cold
    
    if snc_ns.sum() > 0:
        ax.scatter(d['snc_coords'][snc_ns, 0], d['snc_coords'][snc_ns, 1],
                   c='#888888', s=10, alpha=0.7, edgecolors='black',
                   linewidths=0.2, rasterized=True, zorder=3)
    if snc_in_hot.sum() > 0:
        ax.scatter(d['snc_coords'][snc_in_hot, 0], d['snc_coords'][snc_in_hot, 1],
                   c='#D73027', s=14, alpha=0.9, edgecolors='black',
                   linewidths=0.3, rasterized=True, zorder=4)
    if snc_in_cold.sum() > 0:
        ax.scatter(d['snc_coords'][snc_in_cold, 0], d['snc_coords'][snc_in_cold, 1],
                   c='#4575B4', s=14, alpha=0.9, edgecolors='black',
                   linewidths=0.3, rasterized=True, zorder=4)
    
    format_spatial_ax(ax)
    if col == 0:
        ax.set_ylabel('SnC Status', fontsize=6, fontweight='bold', labelpad=4)

# Hide extra colorbar slots
for r in [1, 2]:
    cax_empty = fig.add_subplot(gs[r, n_samples])
    cax_empty.axis('off')

# Legend at bottom
handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#D73027',
           markersize=4, markeredgecolor='black', markeredgewidth=0.2, label='Hot spot (p<.05)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#4575B4',
           markersize=4, markeredgecolor='black', markeredgewidth=0.2, label='Cold spot'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#888888',
           markersize=4, markeredgecolor='black', markeredgewidth=0.2, label='SnC (ns)'),
    Line2D([0], [0], marker='*', color='w', markerfacecolor='black',
           markersize=5, markeredgewidth=0, label='SnC (row 1)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#E8E8E8',
           markersize=3, markeredgewidth=0, label='Non-significant'),
]
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.48, -0.01),
           frameon=True, fancybox=False, edgecolor='#CCC', fontsize=5,
           ncol=5, handletextpad=0.2, borderpad=0.3, columnspacing=0.6)

plt.savefig(str(FIGURES_DIR / 'Fig_gi_star_hotspots.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(str(FIGURES_DIR / 'Fig_gi_star_hotspots.svg'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Fig_gi_star_hotspots.pdf / .svg")

df_hotspot.to_csv(RESULTS_DIR / 'spatial_gi_star_summary.csv', index=False)
print(f"✓ Saved: {RESULTS_DIR / 'spatial_gi_star_summary.csv'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10b-5: NEIGHBORHOOD ENRICHMENT — SnC × CELL TYPE CO-LOCALIZATION
# ═══════════════════════════════════════════════════════════════════════════════
#
# GOAL: Do SnC spots of specific cell types co-localize with each other?
# METHOD: Squidpy nhood_enrichment with 999 permutations on grid graph
#
# Replaces Cell 11 Fisher's test with a spatially-aware permutation test.
# ═══════════════════════════════════════════════════════════════════════════════

print("="*60)
print("10b-5: NEIGHBORHOOD ENRICHMENT — SnC × CELL TYPE")
print("="*60)
print()
print("Question: Do SnC spots of the same/different cell types co-localize?")
print("Method:   Squidpy neighborhood enrichment, 999 permutations")
print()

nhood_results = []

for sid in samples:
    name, _ = get_meta(sid)
    ad = sample_adatas[sid]
    
    ad.obs['ct_sen'] = pd.Categorical(
        ad.obs['dominant_celltype'].astype(str) + '_' +
        np.where(ad.obs['is_senescent'] == 1, 'SnC', 'NonSnC')
    )
    
    sq.gr.nhood_enrichment(ad, cluster_key='ct_sen', n_perms=999)
    
    z_matrix = ad.uns['ct_sen_nhood_enrichment']['zscore']
    cats = ad.obs['ct_sen'].cat.categories.tolist()
    snc_cats = [c for c in cats if c.endswith('_SnC')]
    
    print(f"  {name}: SnC types = {[c.replace('_SnC','') for c in snc_cats]}")
    
    if len(snc_cats) >= 2:
        print(f"    {'Pair':<35s} {'z':>7s} {'':>4s}")
        print(f"    {'─'*48}")
        
        for i, ci in enumerate(cats):
            for j, cj in enumerate(cats):
                if j < i: continue
                if not (ci.endswith('_SnC') or cj.endswith('_SnC')): continue
                z = z_matrix[i, j] if isinstance(z_matrix, np.ndarray) else z_matrix.iloc[i, j]
                if np.isnan(z) or abs(z) <= 1.96: continue
                
                sig = '***' if abs(z) > 3.291 else ('**' if abs(z) > 2.576 else '*')
                pattern = 'attract' if z > 0 else 'repel'
                print(f"    {ci} ↔ {cj:<15s} {z:>+7.2f} {sig:>4s}  ({pattern})")
                
                nhood_results.append({
                    'sample': name, 'label_1': ci, 'label_2': cj,
                    'z_score': z, 'significant': sig, 'pattern': pattern,
                })
    print()

df_nhood = pd.DataFrame(nhood_results) if nhood_results else pd.DataFrame()

# ── Figure: SnC × SnC enrichment heatmaps ────────────────────────────────────
if nhood_results:
    fig, axes = plt.subplots(1, n_samples, figsize=(2.2 * n_samples, 2.5))
    if n_samples == 1:
        axes = [axes]
    
    for col, sid in enumerate(samples):
        name, _ = get_meta(sid)
        ad = sample_adatas[sid]
        z_matrix = ad.uns['ct_sen_nhood_enrichment']['zscore']
        cats = ad.obs['ct_sen'].cat.categories.tolist()
        
        snc_idx = [i for i, c in enumerate(cats) if c.endswith('_SnC')]
        snc_labels = [cats[i].replace('_SnC', '') for i in snc_idx]
        
        ax = axes[col]
        
        if len(snc_idx) < 2:
            ax.text(0.5, 0.5, f'{name}\n<2 SnC types', ha='center', va='center',
                    transform=ax.transAxes, fontsize=7)
            ax.axis('off')
            continue
        
        z_sub = z_matrix[np.ix_(snc_idx, snc_idx)]
        if not isinstance(z_sub, np.ndarray):
            z_sub = z_sub.values
        
        vabs = max(abs(np.nanmin(z_sub)), abs(np.nanmax(z_sub)), 2)
        im = ax.imshow(z_sub, cmap='RdBu_r', vmin=-vabs, vmax=vabs, aspect='auto')
        
        for i in range(z_sub.shape[0]):
            for j in range(z_sub.shape[1]):
                z_val = z_sub[i, j]
                if np.isnan(z_val): continue
                txt_c = 'white' if abs(z_val) > vabs * 0.55 else 'black'
                sig = ''
                if abs(z_val) > 3.291: sig = '\n***'
                elif abs(z_val) > 2.576: sig = '\n**'
                elif abs(z_val) > 1.96: sig = '\n*'
                ax.text(j, i, f'{z_val:+.1f}{sig}', ha='center', va='center',
                        fontsize=4.5, color=txt_c)
        
        ax.set_xticks(range(len(snc_labels)))
        ax.set_xticklabels(snc_labels, fontsize=5, rotation=45, ha='right')
        ax.set_yticks(range(len(snc_labels)))
        ax.set_yticklabels(snc_labels, fontsize=5)
        ax.set_title(name, fontsize=7, fontweight='bold', pad=3)
        
        if col == n_samples - 1:
            cbar = plt.colorbar(im, ax=ax, shrink=0.6, aspect=15, pad=0.06)
            cbar.ax.tick_params(labelsize=4)
            cbar.set_label('z-score', fontsize=5)
    
    plt.tight_layout(w_pad=0.5)
    plt.savefig(str(FIGURES_DIR / 'Fig_nhood_enrichment_snc.pdf'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(str(FIGURES_DIR / 'Fig_nhood_enrichment_snc.svg'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print("✓ Fig_nhood_enrichment_snc.pdf / .svg")

# ── Save ──────────────────────────────────────────────────────────────────────
if not df_nhood.empty:
    df_nhood.to_csv(RESULTS_DIR / 'spatial_nhood_enrichment.csv', index=False)
    print(f"✓ Saved: {RESULTS_DIR / 'spatial_nhood_enrichment.csv'}")

print(f"\n{'='*60}")
print("SPATIAL ANALYSIS COMPLETE")
print(f"{'='*60}")
print()
print("Methods summary:")
print("  10b-1: Spatial neighbor graph (k=6 Visium grid)")
print("  10b-2: Moran's I — global autocorrelation + scatter plots")
print("  10b-3: Join counts — SnC-SnC co-occurrence permutation test")
print("  10b-4: Getis-Ord Gi* — local hot/cold spot detection")
print("  10b-5: Neighborhood enrichment — cell type co-localization")